# Better Distributional Fit Does Not Improve Reachable Coverage Estimation in Fuzzing

This notebook:

1. discovers and loads campaign artifacts
2. builds subject-level artifacts
3. fits parametric models to **observed non-zero incidence-frequency data**
4. computes parametric estimators from the **fitted model parameters**
5. compares parametric and non-parametric estimators
6. saves summary tables and plot data
7. visualises subject-level model fits and estimator behaviour



In [ ]:
from __future__ import annotations

import json
import math
import pickle
import re
from collections import Counter
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.optimize import minimize, root_scalar
from scipy.special import gammaln
from scipy.stats import (
    poisson as sp_poisson,
    expon as sp_expon,
    gamma as sp_gamma,
    nbinom as sp_nbinom,
    chi2 as sp_chi2,
    kstest as sp_ks,
)


In [ ]:
# ============================================================
# Configuration
# ============================================================

RAW_DATA_DIR = Path("/Data")
OUTPUT_DIR = Path("./analysis_outputs")
ARTIFACTS_DIR = OUTPUT_DIR / "campaign_artifacts"
SUBJECT_ARTIFACTS_DIR = OUTPUT_DIR / "subject_artifacts"
SUMMARIES_DIR = OUTPUT_DIR / "summaries"
PLOT_DATA_DIR = SUMMARIES_DIR / "plot_data"
PLOTS_DIR = SUMMARIES_DIR / "plots"
for p in [OUTPUT_DIR, ARTIFACTS_DIR, SUBJECT_ARTIFACTS_DIR, SUMMARIES_DIR, PLOT_DATA_DIR, PLOTS_DIR]:
    p.mkdir(parents=True, exist_ok=True)

REBUILD_CAMPAIGN_ARTIFACTS = False
REBUILD_SUBJECT_ARTIFACTS = False
COVERAGE_KEY = "block_coverage"

RUN_DIR_RE = re.compile(r"^(?P<subject>.+?)_aflpp_run_(?P<run>\d+)$")


KNOWN_TOTALS = {
   
    "parsera": 104952,
    "parserb": 136455,
    "parserc": 231510,
    "parserd": 331301,
    "parsere": 289070,
    
}

print("Artifact directories:")
for p in [OUTPUT_DIR, ARTIFACTS_DIR, SUBJECT_ARTIFACTS_DIR, SUMMARIES_DIR, PLOT_DATA_DIR, PLOTS_DIR]:
    print(f"  {p}")

In [ ]:
# ============================================================
# Raw coverage discovery helpers
# ============================================================

def _natural_sort_key(path: Path) -> List[object]:
    parts = re.split(r"(\d+)", path.name)
    return [int(p) if p.isdigit() else p for p in parts]

def discover_campaigns(data_root: str | Path) -> pd.DataFrame:
    data_root = Path(data_root)
    rows = []

    for p in sorted(data_root.iterdir()):
        if not p.is_dir():
            continue

        m = RUN_DIR_RE.match(p.name)
        if not m:
            continue

        coverage_dir = p / "coverage"
        if not coverage_dir.exists():
            continue

        rows.append({
            "subject": m.group("subject"),
            "run_no": int(m.group("run")),
            "campaign_name": p.name,
            "run_dir": str(p),
            "coverage_dir": str(coverage_dir),
        })

    out = pd.DataFrame(rows)
    if not out.empty:
        out = out.sort_values(["subject", "run_no"]).reset_index(drop=True)
    return out

def get_cov_files_for_campaign(run_dir: str | Path) -> List[Path]:
    coverage_dir = Path(run_dir) / "coverage"
    return sorted(coverage_dir.glob("ft_cov_*.json"), key=_natural_sort_key)

def load_block_coverage(json_path: str | Path, key: str = COVERAGE_KEY) -> np.ndarray:
    with Path(json_path).open("r", encoding="utf-8") as f:
        obj = json.load(f)
    if key not in obj:
        raise KeyError(f"{json_path} does not contain key '{key}'")
    return np.asarray(obj[key], dtype=np.int64)

def pad_vectors(vectors: List[np.ndarray]) -> np.ndarray:
    if len(vectors) == 0:
        return np.empty((0, 0), dtype=np.int64)
    max_len = max(len(v) for v in vectors)
    padded = []
    for v in vectors:
        arr = np.zeros(max_len, dtype=np.int64)
        arr[:len(v)] = v
        padded.append(arr)
    return np.vstack(padded)


In [ ]:
# ============================================================
# Campaign artifacts
# ============================================================

from json import JSONDecodeError

def build_campaign_artifacts(run_dir: str | Path, key: str = COVERAGE_KEY):
    files = get_cov_files_for_campaign(run_dir)
    if len(files) == 0:
        raise FileNotFoundError(f"No coverage JSON files found in {run_dir}")

    cumulative_vectors = []
    for f in files:
        try:
            cumulative_vectors.append(load_block_coverage(f, key=key))
        except JSONDecodeError as e:
            print(f"[BAD JSON FILE] {f}")
            raise JSONDecodeError(f"{e.msg} in file {f}", e.doc, e.pos) from e

    cumulative = pad_vectors(cumulative_vectors)

    prev = np.zeros(cumulative.shape[1], dtype=np.int64)
    incidence_rows = []
    hit_rows = []

    for cur in cumulative:
        delta = np.maximum(cur - prev, 0)
        incidence_rows.append((delta > 0).astype(np.uint8))
        hit_rows.append(delta.astype(np.int64))
        prev = cur

    incidence = np.vstack(incidence_rows)
    hit_counts = np.vstack(hit_rows)
    return incidence, hit_counts, cumulative

def save_campaign_artifact(campaign_row: pd.Series, incidence: np.ndarray, hit_counts: np.ndarray, cumulative: np.ndarray, output_root: Path) -> Path:
    campaign_dir = output_root / campaign_row["campaign_name"]
    campaign_dir.mkdir(parents=True, exist_ok=True)

    with open(campaign_dir / "incidence.pkl", "wb") as f:
        pickle.dump(incidence, f)

    with open(campaign_dir / "hit_counts.pkl", "wb") as f:
        pickle.dump(hit_counts, f)

    metadata = {
        "subject": campaign_row["subject"],
        "run_no": int(campaign_row["run_no"]),
        "campaign_name": campaign_row["campaign_name"],
        "run_dir": campaign_row["run_dir"],
        "n_intervals": int(incidence.shape[0]),
        "n_blocks": int(incidence.shape[1]),
        "final_observed_blocks": int((cumulative[-1] > 0).sum()) if cumulative.size else 0,
    }
    (campaign_dir / "metadata.json").write_text(json.dumps(metadata, indent=2), encoding="utf-8")
    return campaign_dir

def rebuild_all_campaign_artifacts(campaigns_df: pd.DataFrame, output_root: Path, key: str = COVERAGE_KEY) -> pd.DataFrame:
    rows = []

    for _, row in campaigns_df.iterrows():
        incidence, hit_counts, cumulative = build_campaign_artifacts(row["run_dir"], key=key)
        artifact_dir = save_campaign_artifact(row, incidence, hit_counts, cumulative, output_root)
        rows.append({
            "subject": row["subject"],
            "run_no": int(row["run_no"]),
            "campaign_name": row["campaign_name"],
            "artifact_dir": str(artifact_dir),
            "n_intervals": int(incidence.shape[0]),
            "n_blocks": int(incidence.shape[1]),
        })

    return pd.DataFrame(rows)

def load_campaign_artifact(artifact_dir: str | Path) -> Dict[str, object]:
    artifact_dir = Path(artifact_dir)
    with open(artifact_dir / "incidence.pkl", "rb") as f:
        incidence = pickle.load(f)
    with open(artifact_dir / "hit_counts.pkl", "rb") as f:
        hit_counts = pickle.load(f)
    metadata = json.loads((artifact_dir / "metadata.json").read_text(encoding="utf-8"))
    return {"incidence": incidence, "hit_counts": hit_counts, "metadata": metadata}


In [ ]:
# ============================================================
# Build or load campaign artifact index
# ============================================================

campaigns_df = discover_campaigns(RAW_DATA_DIR)
print(f"Discovered campaigns: {len(campaigns_df)}")
display(campaigns_df.head())

artifact_index_path = OUTPUT_DIR / "campaign_artifact_index.csv"

if REBUILD_CAMPAIGN_ARTIFACTS:
    artifact_index_df = rebuild_all_campaign_artifacts(campaigns_df, ARTIFACTS_DIR, key=COVERAGE_KEY)
    artifact_index_df.to_csv(artifact_index_path, index=False)
else:
    artifact_index_df = pd.read_csv(artifact_index_path)
    # repoint to current ARTIFACTS_DIR without rebuilding the .pkl files
    artifact_index_df["artifact_dir"] = artifact_index_df["campaign_name"].map(
        lambda name: str(ARTIFACTS_DIR / name)
    )

n_known = int(artifact_index_df["subject"].isin(KNOWN_TOTALS).sum())
print(f"Campaign artifacts indexed (KNOWN_TOTALS subjects): {n_known}")
display(artifact_index_df.head())

In [ ]:
# ============================================================
# Subject artifacts
# ============================================================

def build_subject_artifact_from_campaigns(subject: str, campaign_group: pd.DataFrame):
    """Build a subject artifacts"""
    hit_sum = None
    campaign_names = []
    run_nos = []
    n_campaigns = 0

    for _, row in campaign_group.sort_values("run_no").iterrows():
        payload = load_campaign_artifact(row["artifact_dir"])
        hit_counts = np.asarray(payload["hit_counts"], dtype=np.float32)
        t, s = hit_counts.shape

        if hit_sum is None:
            hit_sum = np.zeros((t, s), dtype=np.float32)
        else:
            T, S = hit_sum.shape
            if t > T or s > S:
                grown = np.zeros((max(T, t), max(S, s)), dtype=np.float32)
                grown[:T, :S] = hit_sum
                hit_sum = grown

        hit_sum[:t, :s] += hit_counts
        campaign_names.append(row["campaign_name"])
        run_nos.append(int(row["run_no"]))
        n_campaigns += 1

        del payload, hit_counts

    if n_campaigns == 0:
        raise ValueError(f"No campaign artifacts available for subject {subject}")

    # final_hit_count: element-wise average across runs, same shape as campaign hit_counts
    final_hit_count = hit_sum / np.float32(n_campaigns)

    # incidence: derived from final_hit_count — 1 where hit count > 0, else 0
    subject_incidence = (final_hit_count > 0).astype(np.float32)

    metadata = {
        "subject": subject,
        "n_campaigns": n_campaigns,
        "campaign_names": campaign_names,
        "run_nos": run_nos,
        "n_intervals": int(subject_incidence.shape[0]),
        "n_blocks": int(subject_incidence.shape[1]),
        "n_nonzero_final_blocks": int(np.sum(final_hit_count > 0)),
    }
    return subject_incidence, final_hit_count, metadata

def save_subject_artifact(subject: str, subject_incidence: np.ndarray, final_hit_count: np.ndarray, metadata: Dict[str, object], output_root: Path) -> Path:
    subject_dir = output_root / subject
    subject_dir.mkdir(parents=True, exist_ok=True)

    with open(subject_dir / "incidence.pkl", "wb") as f:
        pickle.dump(subject_incidence.astype(np.float32), f, protocol=pickle.HIGHEST_PROTOCOL)
    with open(subject_dir / "final_hit_count.pkl", "wb") as f:
        pickle.dump(final_hit_count.astype(np.float32), f, protocol=pickle.HIGHEST_PROTOCOL)

    (subject_dir / "metadata.json").write_text(json.dumps(metadata, indent=2), encoding="utf-8")
    return subject_dir

def rebuild_all_subject_artifacts(artifact_index_df: pd.DataFrame, output_root: Path) -> pd.DataFrame:
    rows = []
    subjects = [s for s in sorted(artifact_index_df["subject"].unique())]
    n_subjects = len(subjects)
    for i, subject in enumerate(subjects, start=1):
        g = artifact_index_df[artifact_index_df["subject"] == subject]
        subject_incidence, final_hit_count, metadata = build_subject_artifact_from_campaigns(subject, g)
        artifact_dir = save_subject_artifact(subject, subject_incidence, final_hit_count, metadata, output_root)
        rows.append({
            "subject": subject,
            "artifact_dir": str(artifact_dir),
            "n_campaigns": int(metadata["n_campaigns"]),
            "n_intervals": int(metadata["n_intervals"]),
            "n_blocks": int(metadata["n_blocks"]),
            "n_nonzero_final_blocks": int(metadata["n_nonzero_final_blocks"]),
        })
        print(
            f"[{i}/{n_subjects}] Finished subject '{subject}': "
            f"{metadata['n_campaigns']} campaigns, "
            f"{metadata['n_intervals']} intervals x {metadata['n_blocks']} blocks, "
            f"{metadata['n_nonzero_final_blocks']} nonzero blocks -> {artifact_dir}"
        )
        
        del subject_incidence, final_hit_count
    return pd.DataFrame(rows)

def load_subject_artifact(artifact_dir: str | Path) -> Dict[str, object]:
    artifact_dir = Path(artifact_dir)
    with open(artifact_dir / "incidence.pkl", "rb") as f:
        incidence = pickle.load(f)
    with open(artifact_dir / "final_hit_count.pkl", "rb") as f:
        final_hit_count = pickle.load(f)
    metadata = json.loads((artifact_dir / "metadata.json").read_text(encoding="utf-8"))
    return {
        "incidence": np.asarray(incidence, dtype=np.float32),
        "final_hit_count": np.asarray(final_hit_count, dtype=np.float32),
        "metadata": metadata,
    }

In [ ]:
# ============================================================
# Build or load subject artifacts
# ============================================================

subject_artifact_index_path = OUTPUT_DIR / "subject_artifact_index.csv"

if REBUILD_SUBJECT_ARTIFACTS:
    subject_artifact_index_df = rebuild_all_subject_artifacts(artifact_index_df, SUBJECT_ARTIFACTS_DIR)
    subject_artifact_index_df.to_csv(subject_artifact_index_path, index=False)
else:
    subject_artifact_index_df = pd.read_csv(subject_artifact_index_path)

subject_artifacts = {
    row["subject"]: load_subject_artifact(row["artifact_dir"])
    for _, row in subject_artifact_index_df.iterrows()
    
}

print(f"Loaded subject artifacts: {len(subject_artifacts)}")
display(subject_artifact_index_df.head())

In [ ]:

# ============================================================
# Frequency helpers
# ============================================================

def observed_species_from_counts(counts: np.ndarray) -> int:
    """Count the number of blocks (species) observed at least once.
    """
    arr = np.asarray(counts, dtype=float)
    if arr.ndim == 2:
        block_totals = np.nansum(arr, axis=0)
        block_totals = block_totals[np.isfinite(block_totals)]
        return int(np.sum(block_totals > 0))
    # 1D fallback
    arr = arr[np.isfinite(arr)]
    return int(np.sum(arr > 0))

def incidence_frequency_counts(incidence_matrix: np.ndarray) -> Tuple[int, int, int, np.ndarray]:
    arr = np.asarray(incidence_matrix, dtype=float)
    if arr.ndim != 2:
        raise ValueError("incidence_matrix must be 2D")

    q = np.nansum(arr, axis=0).astype(int)
    q = q[q >= 0]

    t = int(arr.shape[0])
    F1 = int(np.sum(q == 1))
    F2 = int(np.sum(q == 2))
    return t, F1, F2, q

def observed_incidence_sample(incidence_matrix: np.ndarray) -> Dict[str, object]:
    t, F1, F2, q = incidence_frequency_counts(incidence_matrix)
    q = np.asarray(q, dtype=int)
    q_pos = q[q > 0]
    values, counts = np.unique(q_pos, return_counts=True) if len(q_pos) else (np.array([], dtype=int), np.array([], dtype=int))
    freq_table = {int(k): int(v) for k, v in zip(values, counts)}
    return {
        "t": t,
        "F1": F1,
        "F2": F2,
        "q": q,
        "q_pos": q_pos,
        "S_obs": int(len(q_pos)),
        "freq_table": freq_table,
    }

def subject_average_nonzero_hit_counts_from_artifact(final_hit_count: np.ndarray) -> np.ndarray:
    arr = np.asarray(final_hit_count, dtype=float)
    if arr.ndim == 2:
        block_totals = np.nansum(arr, axis=0)
    else:
        block_totals = arr
    block_totals = block_totals[np.isfinite(block_totals)]
    return block_totals[block_totals > 0]

def discrete_aic_bic(loglik: float, n_params: int, n: int) -> Tuple[float, float]:
    aic = 2 * n_params - 2 * loglik
    bic = math.log(max(n, 1)) * n_params - 2 * loglik
    return float(aic), float(bic)

def rmse_discrete_counts(x: np.ndarray, pmf_fn) -> float:
    x = np.asarray(x, dtype=int)
    if len(x) == 0:
        return np.nan
    vals, obs = np.unique(x, return_counts=True)
    exp = len(x) * np.asarray([pmf_fn(v) for v in vals], dtype=float)
    return float(np.sqrt(np.mean((obs - exp) ** 2)))

def rmse_binned_continuous(x: np.ndarray, cdf_fn, bins: str | int = "auto") -> float:
    x = np.asarray(x, dtype=float)
    x = x[np.isfinite(x)]
    if len(x) == 0:
        return np.nan
    hist, edges = np.histogram(x, bins=bins)
    cdf_vals = cdf_fn(edges)
    probs = np.maximum(cdf_vals[1:] - cdf_vals[:-1], 0.0)
    exp = len(x) * probs
    return float(np.sqrt(np.mean((hist - exp) ** 2)))

def chi_square_from_expected(obs_counts: np.ndarray, exp_counts: np.ndarray, min_exp: float = 1e-8) -> Tuple[float, float]:
    obs_counts = np.asarray(obs_counts, dtype=float)
    exp_counts = np.asarray(exp_counts, dtype=float)
    exp_counts = np.maximum(exp_counts, min_exp)

    obs_total = obs_counts.sum()
    exp_total = exp_counts.sum()
    if obs_total <= 0 or exp_total <= 0:
        return np.nan, np.nan

    exp_counts = exp_counts * (obs_total / exp_total)
    try:
        stat, p = chisquare(f_obs=obs_counts, f_exp=exp_counts)
        return float(stat), float(p)
    except Exception:
        return np.nan, np.nan

def gof_poisson(x_int: np.ndarray, lam: float) -> Tuple[float, float]:
    values, obs_counts = np.unique(x_int, return_counts=True)
    exp_counts = sp_poisson.pmf(values, mu=lam) * len(x_int)
    return chi_square_from_expected(obs_counts, exp_counts)

def gof_nb(x_int: np.ndarray, size: float, prob: float) -> Tuple[float, float]:
    values, obs_counts = np.unique(x_int, return_counts=True)
    exp_counts = sp_nbinom.pmf(values, n=size, p=prob) * len(x_int)
    return chi_square_from_expected(obs_counts, exp_counts)

def gof_exponential(x: np.ndarray, rate: float) -> Tuple[float, float]:
    try:
        D, p = kstest(x, "expon", args=(0.0, 1.0 / rate))
        return float(D), float(p)
    except Exception:
        return np.nan, np.nan

def gof_gamma(x: np.ndarray, shape: float, scale: float) -> Tuple[float, float]:
    try:
        D, p = kstest(x, "gamma", args=(shape, 0.0, scale))
        return float(D), float(p)
    except Exception:
        return np.nan, np.nan

def best_model_name(df: pd.DataFrame, metric: str, larger_is_better: bool = False):
    if df.empty or metric not in df.columns:
        return np.nan
    tmp = df[["model", metric]].dropna()
    if tmp.empty:
        return np.nan
    if larger_is_better:
        return tmp.sort_values(metric, ascending=False).iloc[0]["model"]
    return tmp.sort_values(metric, ascending=True).iloc[0]["model"]

def _save_plot_csv(df: pd.DataFrame, filename: str, columns=None):
    out_path = PLOT_DATA_DIR / filename
    to_save = df.copy()
    if columns is not None:
        keep_cols = [c for c in columns if c in to_save.columns]
        to_save = to_save[keep_cols]
    to_save.to_csv(out_path, index=False)
    print(f"Saved plot data: {out_path}")
    return out_path


In [ ]:
# ============================================================
# Parametric fits
# ============================================================

def _positive_incidence_counts(incidence_matrix: np.ndarray) -> Dict[str, object]:
    obs = observed_incidence_sample(incidence_matrix)
    q_pos = np.asarray(obs["q_pos"], dtype=float)
    q_pos = q_pos[np.isfinite(q_pos)]
    q_pos = q_pos[q_pos > 0]
    x_int = np.rint(q_pos).astype(int)
    x_int = x_int[x_int > 0]

    return {
        "q_pos": q_pos,
        "x_int": x_int,
        "freq_table": dict(Counter(x_int)),
        "S_obs": int(obs["S_obs"]),
        "F1": int(obs["F1"]),
        "F2": int(obs["F2"]),
        "t": int(obs["t"]),
    }


def _discrete_interval_probs_from_continuous_cdf(vals: np.ndarray, cdf_fn) -> np.ndarray:
    """
    Convert a continuous positive distribution into integer-bin probabilities:
    """
    vals = np.asarray(vals, dtype=int)
    lower = vals.astype(float)
    upper = lower + 1.0
    probs = np.asarray(cdf_fn(upper) - cdf_fn(lower), dtype=float)
    probs = np.clip(probs, 1e-300, 1.0)
    return probs


def rmse_discrete_counts(x_obs: np.ndarray, pmf_fn) -> float:
    vals, counts = np.unique(np.asarray(x_obs, dtype=int), return_counts=True)
    expected = len(x_obs) * np.array([pmf_fn(int(v)) for v in vals], dtype=float)
    return float(np.sqrt(np.mean((counts - expected) ** 2)))


def rmse_binned_continuous(x_obs: np.ndarray, cdf_fn, bins="auto") -> float:
    hist, edges = np.histogram(x_obs, bins=bins)
    probs = np.asarray(cdf_fn(edges[1:]) - cdf_fn(edges[:-1]), dtype=float)
    probs = np.clip(probs, 0.0, 1.0)
    expected = len(x_obs) * probs
    return float(np.sqrt(np.mean((hist - expected) ** 2)))


def discrete_aic_bic(loglik: float, n_params: int, n: int) -> Tuple[float, float]:
    aic = float(2 * n_params - 2 * loglik)
    bic = float(np.log(max(n, 1)) * n_params - 2 * loglik)
    return aic, bic


def gof_poisson(x_int: np.ndarray, lam: float) -> Tuple[float, float]:
    vals, counts = np.unique(x_int, return_counts=True)
    expected = len(x_int) * sp_poisson.pmf(vals, mu=lam)
    expected = np.clip(expected, 1e-12, None)
    chi2 = float(np.sum((counts - expected) ** 2 / expected))
    df = max(len(vals) - 1 - 1, 1)
    p = float(1.0 - sp_chi2.cdf(chi2, df))
    return chi2, p


def gof_nb(x_int: np.ndarray, r: float, p: float) -> Tuple[float, float]:
    vals, counts = np.unique(x_int, return_counts=True)
    expected = len(x_int) * sp_nbinom.pmf(vals, n=r, p=p)
    expected = np.clip(expected, 1e-12, None)
    chi2 = float(np.sum((counts - expected) ** 2 / expected))
    df = max(len(vals) - 1 - 2, 1)
    pval = float(1.0 - sp_chi2.cdf(chi2, df))
    return chi2, pval


def gof_exponential(x: np.ndarray, rate: float) -> Tuple[float, float]:
    ks = sp_ks(x, lambda z: sp_expon.cdf(z, loc=0.0, scale=1.0 / rate))
    return float(ks.statistic), float(ks.pvalue)


def gof_gamma(x: np.ndarray, shape: float, scale: float) -> Tuple[float, float]:
    ks = sp_ks(x, lambda z: sp_gamma.cdf(z, a=shape, loc=0.0, scale=scale))
    return float(ks.statistic), float(ks.pvalue)


def best_model_name(df: pd.DataFrame, metric: str, larger_is_better: bool = False) -> Optional[str]:
    if df.empty or metric not in df.columns:
        return None
    sub = df[["model", metric]].dropna()
    if sub.empty:
        return None
    idx = sub[metric].idxmax() if larger_is_better else sub[metric].idxmin()
    return str(sub.loc[idx, "model"])


# ------------------------------------------------------------
# Incidence-frequency models used as estimators
# ------------------------------------------------------------

def fit_poisson_incidence_model(incidence_matrix: np.ndarray) -> Dict[str, float]:
    dat = _positive_incidence_counts(incidence_matrix)
    x_int = dat["x_int"]
    S_obs = dat["S_obs"]
    if len(x_int) == 0:
        return {}

    sample_mean = float(np.mean(x_int))

    def f(lam: float) -> float:
        denom = 1.0 - math.exp(-lam)
        return lam / denom - sample_mean

    upper = max(10.0, sample_mean * 10.0)
    while f(upper) < 0:
        upper *= 2.0
        if upper > 1e10:
            break

    try:
        root = root_scalar(f, bracket=[1e-12, upper], method="brentq")
        lam_hat = float(root.root)
    except Exception:
        lam_hat = max(sample_mean, 1e-12)

    p0 = float(math.exp(-lam_hat))
    p_seen = float(max(1.0 - p0, 1e-12))
    S_hat = float(S_obs / p_seen)

    loglik = float(np.sum(sp_poisson.logpmf(x_int, mu=lam_hat) - math.log(p_seen)))
    aic, bic = discrete_aic_bic(loglik, n_params=1, n=len(x_int))

    def zt_pois_pmf(k: int) -> float:
        if k <= 0:
            return 0.0
        return float(sp_poisson.pmf(k, mu=lam_hat) / p_seen)

    rmse = rmse_discrete_counts(x_int, zt_pois_pmf)

    return {
        "model": "Poisson",
        "fit_family": "zero-truncated Poisson on incidence frequencies",
        "n_obs": int(len(x_int)),
        "S_obs": S_obs,
        "lambda": lam_hat,
        "p0": p0,
        "p_seen": p_seen,
        "estimate": S_hat,
        "unseen": float(max(S_hat - S_obs, 0.0)),
        "loglik": loglik,
        "aic": aic,
        "bic": bic,
        "rmse": rmse,
    }


def fit_exponential_incidence_model(incidence_matrix: np.ndarray) -> Dict[str, float]:
    """
    Discretized zero-truncated Exponential on integer incidence counts.
    """
    dat = _positive_incidence_counts(incidence_matrix)
    x = dat["q_pos"]
    x_int = dat["x_int"]
    S_obs = dat["S_obs"]
    if len(x) == 0:
        return {}

    rate0 = 1.0 / max(float(np.mean(x)), 1e-12)

    def negloglik(log_rate_arr: np.ndarray) -> float:
        rate_ = float(np.exp(log_rate_arr[0]))
        scale_ = 1.0 / rate_
        p0_ = float(sp_expon.cdf(1.0, loc=0.0, scale=scale_) - sp_expon.cdf(0.0, loc=0.0, scale=scale_))
        p_seen_ = max(1.0 - p0_, 1e-300)
        probs = _discrete_interval_probs_from_continuous_cdf(
            x_int,
            lambda z: sp_expon.cdf(z, loc=0.0, scale=scale_),
        )
        ll = np.sum(np.log(probs) - math.log(p_seen_))
        return -float(ll) if np.isfinite(ll) else 1e100

    res = minimize(
        negloglik,
        x0=np.array([math.log(max(rate0, 1e-12))], dtype=float),
        method="L-BFGS-B",
        bounds=[(-20, 20)],
    )
    if not res.success:
        return {}

    rate_hat = float(np.exp(res.x[0]))
    scale_hat = float(1.0 / rate_hat)
    p0 = float(sp_expon.cdf(1.0, loc=0.0, scale=scale_hat) - sp_expon.cdf(0.0, loc=0.0, scale=scale_hat))
    p_seen = float(max(1.0 - p0, 1e-12))
    S_hat = float(S_obs / p_seen)

    probs = _discrete_interval_probs_from_continuous_cdf(
        x_int,
        lambda z: sp_expon.cdf(z, loc=0.0, scale=scale_hat),
    )
    loglik = float(np.sum(np.log(probs) - math.log(p_seen)))
    aic, bic = discrete_aic_bic(loglik, n_params=1, n=len(x_int))

    def zt_exp_pmf(k: int) -> float:
        if k <= 0:
            return 0.0
        pk = float(sp_expon.cdf(k + 1.0, loc=0.0, scale=scale_hat) - sp_expon.cdf(k, loc=0.0, scale=scale_hat))
        return pk / p_seen

    rmse = rmse_discrete_counts(x_int, zt_exp_pmf)

    return {
        "model": "Exponential",
        "fit_family": "discretized zero-truncated Exponential on incidence frequencies",
        "n_obs": int(len(x_int)),
        "S_obs": S_obs,
        "rate": rate_hat,
        "scale": scale_hat,
        "mean": scale_hat,
        "p0": p0,
        "p_seen": p_seen,
        "estimate": S_hat,
        "unseen": float(max(S_hat - S_obs, 0.0)),
        "loglik": loglik,
        "aic": aic,
        "bic": bic,
        "rmse": rmse,
    }


def fit_gamma_incidence_model(incidence_matrix: np.ndarray) -> Dict[str, float]:
    """
    Discretized zero-truncated Gamma on integer incidence counts.
    """
    dat = _positive_incidence_counts(incidence_matrix)
    x = dat["q_pos"]
    x_int = dat["x_int"]
    S_obs = dat["S_obs"]
    if len(x) == 0:
        return {}

    try:
        shape0, loc0, scale0 = sp_gamma.fit(x, floc=0.0)
    except Exception:
        return {}
    shape0 = float(max(shape0, 1e-6))
    scale0 = float(max(scale0, 1e-6))

    def negloglik(log_params: np.ndarray) -> float:
        shape_ = float(np.exp(log_params[0]))
        scale_ = float(np.exp(log_params[1]))
        p0_ = float(sp_gamma.cdf(1.0, a=shape_, loc=0.0, scale=scale_) - sp_gamma.cdf(0.0, a=shape_, loc=0.0, scale=scale_))
        p_seen_ = max(1.0 - p0_, 1e-300)
        probs = _discrete_interval_probs_from_continuous_cdf(
            x_int,
            lambda z: sp_gamma.cdf(z, a=shape_, loc=0.0, scale=scale_),
        )
        ll = np.sum(np.log(probs) - math.log(p_seen_))
        return -float(ll) if np.isfinite(ll) else 1e100

    res = minimize(
        negloglik,
        x0=np.array([math.log(shape0), math.log(scale0)], dtype=float),
        method="L-BFGS-B",
        bounds=[(-20, 20), (-20, 20)],
    )
    if not res.success:
        return {}

    shape_hat = float(np.exp(res.x[0]))
    scale_hat = float(np.exp(res.x[1]))
    rate_hat = float(1.0 / scale_hat)

    p0 = float(sp_gamma.cdf(1.0, a=shape_hat, loc=0.0, scale=scale_hat) - sp_gamma.cdf(0.0, a=shape_hat, loc=0.0, scale=scale_hat))
    p_seen = float(max(1.0 - p0, 1e-12))
    S_hat = float(S_obs / p_seen)

    probs = _discrete_interval_probs_from_continuous_cdf(
        x_int,
        lambda z: sp_gamma.cdf(z, a=shape_hat, loc=0.0, scale=scale_hat),
    )
    loglik = float(np.sum(np.log(probs) - math.log(p_seen)))
    aic, bic = discrete_aic_bic(loglik, n_params=2, n=len(x_int))

    def zt_gamma_pmf(k: int) -> float:
        if k <= 0:
            return 0.0
        pk = float(
            sp_gamma.cdf(k + 1.0, a=shape_hat, loc=0.0, scale=scale_hat)
            - sp_gamma.cdf(k, a=shape_hat, loc=0.0, scale=scale_hat)
        )
        return pk / p_seen

    rmse = rmse_discrete_counts(x_int, zt_gamma_pmf)

    return {
        "model": "Gamma",
        "fit_family": "discretized zero-truncated Gamma on incidence frequencies",
        "n_obs": int(len(x_int)),
        "S_obs": S_obs,
        "shape": shape_hat,
        "scale": scale_hat,
        "rate": rate_hat,
        "mean": float(shape_hat * scale_hat),
        "p0": p0,
        "p_seen": p_seen,
        "estimate": S_hat,
        "unseen": float(max(S_hat - S_obs, 0.0)),
        "loglik": loglik,
        "aic": aic,
        "bic": bic,
        "rmse": rmse,
    }


def fit_nb_incidence_model(incidence_matrix: np.ndarray) -> Dict[str, float]:
    dat = _positive_incidence_counts(incidence_matrix)
    x_int = dat["x_int"]
    freq_table = dat["freq_table"]
    S_obs = dat["S_obs"]
    if len(x_int) == 0:
        return {}

    mean_x = float(np.mean(x_int))
    var_x = float(np.var(x_int, ddof=1)) if len(x_int) > 1 else mean_x + 1.0

    if var_x > mean_x:
        size0 = max((mean_x ** 2) / max(var_x - mean_x, 1e-9), 1e-3)
    else:
        size0 = 1e3
    prob0 = float(np.clip(size0 / (size0 + mean_x), 1e-6, 1.0 - 1e-6))

    def neg_loglik(log_params: np.ndarray) -> float:
        size = math.exp(log_params[0])
        prob = 1.0 / (1.0 + math.exp(-log_params[1]))
        p0 = float(sp_nbinom.pmf(0, n=size, p=prob))
        if (not np.isfinite(p0)) or p0 >= 1.0:
            return 1e100

        log_seen = math.log(max(1.0 - p0, 1e-300))
        total = 0.0
        for k, fk in freq_table.items():
            if k >= 1:
                ll_k = float(sp_nbinom.logpmf(k, n=size, p=prob) - log_seen)
                if not np.isfinite(ll_k):
                    return 1e100
                total += fk * ll_k
        return -float(total)

    res = minimize(
        neg_loglik,
        x0=np.array([math.log(size0), math.log(prob0 / (1.0 - prob0))], dtype=float),
        method="L-BFGS-B",
        bounds=[(-20, 20), (-20, 20)],
    )

    if not res.success:
        return {}

    size_hat = float(math.exp(res.x[0]))
    prob_hat = float(1.0 / (1.0 + math.exp(-res.x[1])))
    p0 = float(sp_nbinom.pmf(0, n=size_hat, p=prob_hat))
    p_seen = float(max(1.0 - p0, 1e-12))
    S_hat = float(S_obs / p_seen)
    mu_hat = float(size_hat * (1.0 - prob_hat) / prob_hat)

    loglik = -float(res.fun)
    aic, bic = discrete_aic_bic(loglik, n_params=2, n=len(x_int))

    def zt_nb_pmf(k: int) -> float:
        if k <= 0:
            return 0.0
        return float(sp_nbinom.pmf(k, n=size_hat, p=prob_hat) / p_seen)

    rmse = rmse_discrete_counts(x_int, zt_nb_pmf)

    return {
        "model": "Negative Binomial",
        "fit_family": "zero-truncated Negative Binomial on incidence frequencies",
        "n_obs": int(len(x_int)),
        "S_obs": S_obs,
        "size": size_hat,
        "prob": prob_hat,
        "mean": mu_hat,
        "p0": p0,
        "p_seen": p_seen,
        "estimate": S_hat,
        "unseen": float(max(S_hat - S_obs, 0.0)),
        "loglik": loglik,
        "aic": aic,
        "bic": bic,
        "rmse": rmse,
    }


def fit_gamma_poisson_incidence_model(incidence_matrix: np.ndarray) -> Dict[str, float]:
    obs = observed_incidence_sample(incidence_matrix)
    q_pos = np.asarray(obs["q_pos"], dtype=int)
    freq_table = obs["freq_table"]
    S_obs = int(obs["S_obs"])
    if len(q_pos) == 0:
        return {}

    def gp_logpmf(k: int, alpha: float, beta: float) -> float:
        return (
            alpha * math.log(beta)
            + gammaln(k + alpha)
            - gammaln(alpha)
            - gammaln(k + 1)
            - (k + alpha) * math.log(beta + 1.0)
        )

    def neg_loglik(log_params: np.ndarray) -> float:
        alpha = math.exp(log_params[0])
        beta = math.exp(log_params[1])

        p0 = (beta / (beta + 1.0)) ** alpha
        if not np.isfinite(p0) or p0 >= 1.0:
            return 1e100

        log_one_minus_p0 = math.log(max(1.0 - p0, 1e-300))
        total = 0.0
        for k, fk in freq_table.items():
            if k >= 1:
                ll_k = gp_logpmf(k, alpha, beta) - log_one_minus_p0
                if not np.isfinite(ll_k):
                    return 1e100
                total += fk * ll_k
        return -float(total)

    mean_x = float(np.mean(q_pos))
    var_x = float(np.var(q_pos, ddof=1)) if len(q_pos) > 1 else mean_x + 1.0

    if var_x > mean_x:
        alpha0 = max(mean_x**2 / (var_x - mean_x), 1e-3)
    else:
        alpha0 = 1e3
    beta0 = max(alpha0 / mean_x, 1e-3)

    res = minimize(
        neg_loglik,
        x0=np.array([math.log(alpha0), math.log(beta0)], dtype=float),
        method="L-BFGS-B",
        bounds=[(-20, 20), (-20, 20)],
    )

    if not res.success:
        return {}

    alpha_hat = float(math.exp(res.x[0]))
    beta_hat = float(math.exp(res.x[1]))

    p0 = float((beta_hat / (beta_hat + 1.0)) ** alpha_hat)
    p_seen = float(max(1.0 - p0, 1e-12))
    S_hat = float(S_obs / p_seen)

    loglik = -float(res.fun)
    aic, bic = discrete_aic_bic(loglik, n_params=2, n=len(q_pos))

    def zt_gp_pmf(k: int) -> float:
        if k <= 0:
            return 0.0
        logp = gp_logpmf(k, alpha_hat, beta_hat) - math.log(p_seen)
        return float(math.exp(logp))

    rmse = rmse_discrete_counts(q_pos, zt_gp_pmf)

    return {
        "model": "Gamma-Poisson",
        "fit_family": "zero-truncated Gamma-Poisson on incidence frequencies",
        "n_obs": int(len(q_pos)),
        "S_obs": S_obs,
        "shape": alpha_hat,
        "rate": beta_hat,
        "scale": float(1.0 / beta_hat),
        "mean": float(alpha_hat / beta_hat),
        "p0": p0,
        "p_seen": p_seen,
        "estimate": S_hat,
        "unseen": float(max(S_hat - S_obs, 0.0)),
        "loglik": loglik,
        "aic": aic,
        "bic": bic,
        "rmse": rmse,
    }


def fit_zipf_mandelbrot_model(x: np.ndarray, ranks: Optional[np.ndarray] = None) -> Dict[str, float]:
    x = np.asarray(x, dtype=float)
    x = x[np.isfinite(x)]
    x = x[x > 0]
    if len(x) == 0:
        return {}

    if ranks is None:
        sorted_counts = np.sort(x)[::-1]
        ranks = np.arange(1, len(sorted_counts) + 1, dtype=float)
    else:
        sorted_counts = np.asarray(x, dtype=float)
        ranks = np.asarray(ranks, dtype=float)
        if len(ranks) != len(sorted_counts):
            raise ValueError(f"Length mismatch: got {len(ranks)} ranks for {len(sorted_counts)} counts")

    n = len(sorted_counts)
    J = float(np.sum(sorted_counts))

    def loss(params: np.ndarray) -> float:
        s, q = params
        if s <= 0.0 or q < 0.0:
            return 1e18
        weights = (ranks + q) ** (-s)
        C = 1.0 / np.sum(weights)
        pred = J * C * weights
        return float(np.sum((sorted_counts - pred) ** 2))
        #return float(np.sum((np.log(sorted_counts) - np.log(pred)) ** 2))

    res = minimize(
        loss,
        x0=np.array([1.0, 1.0]),
        bounds=[(1e-6, 10.0), (0.0, 100.0)],
        method="L-BFGS-B",
    )

    if not res.success:
        s_hat, q_hat = 1.0, 1.0
    else:
        s_hat, q_hat = float(res.x[0]), float(res.x[1])

    weights = (ranks + q_hat) ** (-s_hat)
    C = 1.0 / np.sum(weights)
    pred = J * C * weights

    #residuals = np.log(sorted_counts) - np.log(pred)
    residuals = sorted_counts - pred
    sigma2 = float(np.mean(residuals ** 2))
    if sigma2 <= 0.0 or not np.isfinite(sigma2):
        sigma2 = 1.0

    loglik = -0.5 * n * (np.log(2.0 * np.pi * sigma2) + 1.0)
    aic, bic = discrete_aic_bic(loglik, n_params=2, n=n)
    rmse = float(np.sqrt(np.mean((sorted_counts - pred) ** 2)))
    
    return {
        "model": "Zipf-Mandelbrot",
        "fit_family": "Zipf-Mandelbrot on rank-frequency data derived from incidence sums",
        "n_obs": int(n),
        "S_obs": int(n),
        "J": J,
        "C": float(C),
        "s": s_hat,
        "q": q_hat,
        "pred": pred,
        "ranks": ranks,
        "loglik": loglik,
        "aic": aic,
        "bic": bic,
        "rmse": rmse,
    }


# ------------------------------------------------------------
# Hit-count models
# ------------------------------------------------------------

def fit_poisson_hitcount_model(x_avg: np.ndarray) -> Dict[str, float]:
    x_int = np.rint(np.asarray(x_avg, dtype=float)).astype(int)
    x_int = x_int[np.isfinite(x_int)]
    x_int = x_int[x_int >= 0]
    if len(x_int) == 0:
        return {}

    lam = float(np.mean(x_int))
    loglik = float(np.sum(sp_poisson.logpmf(x_int, mu=lam)))
    aic, bic = discrete_aic_bic(loglik, n_params=1, n=len(x_int))
    rmse = rmse_discrete_counts(x_int, lambda v: sp_poisson.pmf(v, mu=lam))
    chi2, chi2_p = gof_poisson(x_int, lam)

    return {
        "model": "Poisson",
        "fit_family": "Poisson on nonzero averaged hit counts",
        "n_obs": int(len(x_avg)),
        "n_obs_discrete": int(len(x_int)),
        "lambda": lam,
        "loglik": loglik,
        "aic": aic,
        "bic": bic,
        "rmse": rmse,
        "gof_stat": chi2,
        "gof_p": chi2_p,
        "gof_name": "Chi-square",
    }


def fit_exponential_hitcount_model(x_avg: np.ndarray) -> Dict[str, float]:
    x = np.asarray(x_avg, dtype=float)
    x = x[np.isfinite(x)]
    x = x[x > 0]
    if len(x) == 0:
        return {}

    rate = 1.0 / float(np.mean(x))
    loglik = float(np.sum(sp_expon.logpdf(x, loc=0.0, scale=1.0 / rate)))
    aic, bic = discrete_aic_bic(loglik, n_params=1, n=len(x))
    rmse = rmse_binned_continuous(x, lambda z: sp_expon.cdf(z, loc=0.0, scale=1.0 / rate))
    ks_D, ks_p = gof_exponential(x, rate)

    return {
        "model": "Exponential",
        "fit_family": "Exponential on nonzero averaged hit counts",
        "n_obs": int(len(x)),
        "rate": float(rate),
        "loglik": loglik,
        "aic": aic,
        "bic": bic,
        "rmse": rmse,
        "gof_stat": ks_D,
        "gof_p": ks_p,
        "gof_name": "KS",
    }


def fit_gamma_hitcount_model(x_avg: np.ndarray) -> Dict[str, float]:
    x = np.asarray(x_avg, dtype=float)
    x = x[np.isfinite(x)]
    x = x[x > 0]
    if len(x) == 0:
        return {}

    try:
        shape, loc, scale = sp_gamma.fit(x, floc=0.0)
    except Exception:
        return {}
    shape = float(shape)
    scale = float(scale)
    rate = float(1.0 / scale) if scale > 0 else np.nan

    loglik = float(np.sum(sp_gamma.logpdf(x, a=shape, loc=0.0, scale=scale)))
    aic, bic = discrete_aic_bic(loglik, n_params=2, n=len(x))
    rmse = rmse_binned_continuous(x, lambda z: sp_gamma.cdf(z, a=shape, loc=0.0, scale=scale))
    ks_D, ks_p = gof_gamma(x, shape, scale)

    return {
        "model": "Gamma",
        "fit_family": "Gamma on nonzero averaged hit counts",
        "n_obs": int(len(x)),
        "shape": shape,
        "scale": scale,
        "rate": rate,
        "mean": float(shape * scale),
        "loglik": loglik,
        "aic": aic,
        "bic": bic,
        "rmse": rmse,
        "gof_stat": ks_D,
        "gof_p": ks_p,
        "gof_name": "KS",
    }


def fit_nb_hitcount_model(x_avg: np.ndarray) -> Dict[str, float]:
    x = np.asarray(x_avg, dtype=float)
    x = x[np.isfinite(x)]
    x = x[x > 0]
    if len(x) == 0:
        return {}

    x_int = np.round(x).astype(np.int64)
    x_int = x_int[x_int > 0]
    if len(x_int) == 0:
        return {}

    mean_x = float(np.mean(x_int))
    var_x = float(np.var(x_int, ddof=1)) if len(x_int) > 1 else mean_x + 1.0

    if var_x <= mean_x:
        r0 = 1000.0
    else:
        r0 = mean_x**2 / (var_x - mean_x)

    r0 = float(np.clip(r0, 1e-6, 1e6))
    p0 = float(np.clip(r0 / (r0 + mean_x), 1e-10, 1 - 1e-10))

    def negloglik(params):
        log_r, logit_p = params
        r = float(np.clip(np.exp(log_r), 1e-6, 1e6))
        p = float(np.clip(1.0 / (1.0 + np.exp(-logit_p)), 1e-10, 1 - 1e-10))
        try:
            ll_vals = sp_nbinom.logpmf(x_int, n=r, p=p)
            if not np.all(np.isfinite(ll_vals)):
                return 1e100
            ll = float(np.sum(ll_vals))
            return -ll if np.isfinite(ll) else 1e100
        except Exception:
            return 1e100

    res = minimize(
    negloglik,
    x0=np.array([np.log(r0), np.log(p0 / (1 - p0))], dtype=float),
    method="L-BFGS-B",
    bounds=[(-20, 20), (-20, 20)],
)

    if not res.success:
        # fallback: use initial-guess parameters instead of silently dropping the row
        log_r0 = float(np.clip(np.log(r0), -20, 20))
        logit_p0 = float(np.clip(np.log(p0 / (1 - p0)), -20, 20))
        r_hat = float(np.exp(log_r0))
        p_hat = float(1.0 / (1.0 + np.exp(-logit_p0)))
        loglik = float(np.sum(sp_nbinom.logpmf(x_int, n=r_hat, p=p_hat)))
    else:
        r_hat = float(np.clip(np.exp(res.x[0]), 1e-6, 1e6))
        p_hat = float(np.clip(1.0 / (1.0 + np.exp(-res.x[1])), 1e-10, 1 - 1e-10))
        loglik = -float(res.fun)

    mu_hat = float(r_hat * (1 - p_hat) / p_hat)
    aic, bic = discrete_aic_bic(loglik, n_params=2, n=len(x_int))
    rmse = rmse_discrete_counts(x_int, lambda v: sp_nbinom.pmf(v, n=r_hat, p=p_hat))
    chi2, chi2_p = gof_nb(x_int, r_hat, p_hat)

    return {
        "model": "Negative Binomial",
        "fit_family": "Negative Binomial on nonzero averaged hit counts",
        "n_obs": int(len(x)),
        "n_obs_discrete": int(len(x_int)),
        "size": r_hat,
        "prob": p_hat,
        "mu": mu_hat,
        "loglik": loglik,
        "aic": aic,
        "bic": bic,
        "rmse": rmse,
        "gof_stat": chi2,
        "gof_p": chi2_p,
        "gof_name": "Chi-square",
    }


def fit_all_hitcount_models(x_avg: np.ndarray) -> pd.DataFrame:
    rows = []
    for fn in [
        fit_poisson_hitcount_model,
        fit_exponential_hitcount_model,
        fit_gamma_hitcount_model,
        fit_nb_hitcount_model,
    ]:
        res = fn(x_avg)
        if res:
            rows.append(res)
    return pd.DataFrame(rows)


def fit_all_incidence_models(incidence_matrix: np.ndarray) -> pd.DataFrame:
    rows = []
    for fn in [
        fit_poisson_incidence_model,
        fit_exponential_incidence_model,
        fit_gamma_incidence_model,
        fit_nb_incidence_model,
        fit_gamma_poisson_incidence_model,
    ]:
        res = fn(incidence_matrix)
        if res:
            rows.append(res)
    return pd.DataFrame(rows)


In [ ]:

# ============================================================
# Subject-level fitted model table
# ============================================================

def canonical_fit_model_name(model_name: str) -> str:
    model_name = str(model_name).strip()
    aliases = {
        "NegBin": "Negative Binomial",
        "Negative Binomial": "Negative Binomial",
        "Poisson": "Poisson",
        "Exponential": "Exponential",
        "Gamma": "Gamma",
        "Gamma-Poisson": "Gamma-Poisson",
        "Zipf-Mandelbrot": "Zipf-Mandelbrot",
    }
    return aliases.get(model_name, model_name)


def explicit_model_label(base_model: str, fit_input: str) -> str:
    base_model = canonical_fit_model_name(base_model)

    label_map = {
        ("Poisson", "incidence_frequency"): "Poisson (incidence-frequency)",
        ("Exponential", "incidence_frequency"): "Exponential (incidence-frequency)",
        ("Gamma", "incidence_frequency"): "Gamma (incidence-frequency)",
        ("Negative Binomial", "incidence_frequency"): "Negative Binomial (incidence-frequency)",
        ("Gamma-Poisson", "incidence_frequency"): "Gamma-Poisson (incidence-frequency)",

        ("Poisson", "nonzero_averaged_hit_count"): "Poisson (hit-count)",
        ("Exponential", "nonzero_averaged_hit_count"): "Exponential (hit-count)",
        ("Gamma", "nonzero_averaged_hit_count"): "Gamma (hit-count)",
        ("Negative Binomial", "nonzero_averaged_hit_count"): "Negative Binomial (hit-count)",

        ("Zipf-Mandelbrot", "incidence_sum_axis1"): "Zipf-Mandelbrot (coverage-frequency)",

    }
    return label_map.get((base_model, fit_input), f"{base_model} ({fit_input})")


fit_rows = []

for subject, payload in sorted(subject_artifacts.items()):
    incidence = np.asarray(payload["incidence"], dtype=np.float32)
    final_hit_count = np.asarray(payload["final_hit_count"], dtype=np.float32)

    # --------------------------------------------------------
    # Incidence-frequency fits
    # --------------------------------------------------------
    incidence_fit_df = fit_all_incidence_models(incidence)
    if not incidence_fit_df.empty:
        incidence_fit_df = incidence_fit_df.copy()
        incidence_fit_df.insert(0, "subject", subject)
        incidence_fit_df["fit_input"] = "incidence_frequency"
        incidence_fit_df["base_model"] = incidence_fit_df["model"].astype(str).map(canonical_fit_model_name)
        incidence_fit_df["model"] = incidence_fit_df["base_model"].apply(
            lambda m: explicit_model_label(m, "incidence_frequency")
        )
        incidence_fit_df["best_by_rmse"] = best_model_name(incidence_fit_df, "rmse")
        incidence_fit_df["best_by_aic"] = best_model_name(incidence_fit_df, "aic")
        incidence_fit_df["best_by_bic"] = best_model_name(incidence_fit_df, "bic")
        incidence_fit_df["best_by_loglik"] = best_model_name(
            incidence_fit_df, "loglik", larger_is_better=True
        )
        fit_rows.extend(incidence_fit_df.to_dict(orient="records"))

    # --------------------------------------------------------
    # Hit-count fits
    # --------------------------------------------------------
    x_avg = subject_average_nonzero_hit_counts_from_artifact(final_hit_count)
    hitcount_fit_df = fit_all_hitcount_models(x_avg)
    if not hitcount_fit_df.empty:
        hitcount_fit_df = hitcount_fit_df.copy()
        hitcount_fit_df.insert(0, "subject", subject)
        hitcount_fit_df["fit_input"] = "nonzero_averaged_hit_count"
        hitcount_fit_df["base_model"] = hitcount_fit_df["model"].astype(str).map(canonical_fit_model_name)
        hitcount_fit_df["model"] = hitcount_fit_df["base_model"].apply(
            lambda m: explicit_model_label(m, "nonzero_averaged_hit_count")
        )
        hitcount_fit_df["best_by_rmse"] = best_model_name(hitcount_fit_df, "rmse")
        hitcount_fit_df["best_by_aic"] = best_model_name(hitcount_fit_df, "aic")
        hitcount_fit_df["best_by_bic"] = best_model_name(hitcount_fit_df, "bic")
        hitcount_fit_df["best_by_loglik"] = best_model_name(
            hitcount_fit_df, "loglik", larger_is_better=True
        )
        fit_rows.extend(hitcount_fit_df.to_dict(orient="records"))

    # --------------------------------------------------------
    # Zipf-Mandelbrot on incidence-sum rank-frequency
    # --------------------------------------------------------
    zipf_counts = np.asarray(payload["incidence"], dtype=float).sum(axis=0)
    zipf_counts = zipf_counts[np.isfinite(zipf_counts)]
    zipf_counts = zipf_counts[zipf_counts > 0]

    if len(zipf_counts) > 0:
        zipf = fit_zipf_mandelbrot_model(np.sort(zipf_counts)[::-1])
        if zipf:
            zipf["base_model"] = "Zipf-Mandelbrot"
            zipf["model"] = explicit_model_label("Zipf-Mandelbrot", "incidence_sum_axis1")
            fit_rows.append({"subject": subject, **zipf, "fit_input": "incidence_sum_axis1"})

  
    
subject_fit_df = (
    pd.DataFrame(fit_rows)
    .sort_values(["subject", "fit_input", "base_model", "model"])
    .reset_index(drop=True)
)

subject_fit_df.to_csv(SUMMARIES_DIR / "subject_parametric_fit_summary.csv", index=False)
subject_fit_df[subject_fit_df["fit_input"] == "incidence_frequency"].to_csv(
    SUMMARIES_DIR / "subject_parametric_fit_summary_incidence_frequency.csv", index=False
)
subject_fit_df[subject_fit_df["fit_input"] == "nonzero_averaged_hit_count"].to_csv(
    SUMMARIES_DIR / "subject_parametric_fit_summary_hit_count.csv", index=False
)
subject_fit_df[subject_fit_df["fit_input"] == "incidence_sum_axis1"].to_csv(
    SUMMARIES_DIR / "subject_parametric_fit_summary_zipf_coverage_frequency.csv", index=False
)

print("Saved:", SUMMARIES_DIR / "subject_parametric_fit_summary.csv")
print("Saved:", SUMMARIES_DIR / "subject_parametric_fit_summary_incidence_frequency.csv")
print("Saved:", SUMMARIES_DIR / "subject_parametric_fit_summary_hit_count.csv")
print("Saved:", SUMMARIES_DIR / "subject_parametric_fit_summary_zipf_coverage_frequency.csv")
display(subject_fit_df.head(30))


In [ ]:
# ============================================================
# Subject-level fit plots
# ============================================================

def _save_subject_fit_plot_csv(df: pd.DataFrame, filename: str):
    outpath = PLOTS_DIR / filename
    df.to_csv(outpath, index=False)


def plot_poisson_fit(subject: str, incidence_matrix: np.ndarray, save_csv: bool = True):
    obs = observed_incidence_sample(incidence_matrix)
    q_pos = np.asarray(obs["q_pos"], dtype=int)
    fit = fit_poisson_incidence_model(incidence_matrix)
    if len(q_pos) == 0 or not fit:
        print(f"No positive incidence counts for {subject}")
        return

    vals, counts = np.unique(q_pos, return_counts=True)
    fitted = len(q_pos) * np.array([
        sp_poisson.pmf(v, mu=fit["lambda"]) / max(1.0 - fit["p0"], 1e-12)
        for v in vals
    ], dtype=float)

    plot_df = pd.DataFrame({
        "subject": subject,
        "k": vals,
        "observed_count": counts,
        "fitted_count": fitted,
        "model": "Poisson (incidence-frequency)",
        "fit_input": "incidence_frequency",
    })
    if save_csv:
        _save_subject_fit_plot_csv(plot_df, f"{subject}_poisson_incidence_fit.csv")

    plt.figure(figsize=(6, 5))
    plt.scatter(vals, counts, s=30, label="Observed")
    plt.plot(vals, fitted, label=f"Poisson fit (lambda={fit['lambda']:.3f})")
    plt.xlabel("Incidence frequency k")
    plt.ylabel("Number of coverage elements")
    plt.title(f"Poisson incidence fit: {subject}")
    plt.legend()
    plt.tight_layout()
    plt.show()


def plot_exponential_incidence_fit(subject: str, incidence_matrix: np.ndarray, save_csv: bool = True):
    dat = _positive_incidence_counts(incidence_matrix)
    x_int = dat["x_int"]
    fit = fit_exponential_incidence_model(incidence_matrix)
    if len(x_int) == 0 or not fit:
        print(f"No positive incidence counts for {subject}")
        return

    vals, counts = np.unique(x_int, return_counts=True)

    fitted = len(x_int) * np.array([
        (
            sp_expon.cdf(v + 1.0, loc=0.0, scale=fit["scale"])
            - sp_expon.cdf(v, loc=0.0, scale=fit["scale"])
        ) / max(fit["p_seen"], 1e-12)
        for v in vals
    ], dtype=float)

    plot_df = pd.DataFrame({
        "subject": subject,
        "k": vals,
        "observed_count": counts,
        "fitted_count": fitted,
        "model": "Exponential (incidence-frequency)",
        "fit_input": "incidence_frequency",
    })
    if save_csv:
        _save_subject_fit_plot_csv(plot_df, f"{subject}_exponential_incidence_fit.csv")

    plt.figure(figsize=(6, 5))
    plt.scatter(vals, counts, s=30, label="Observed")
    plt.plot(vals, fitted, label=f"Exponential fit (rate={fit['rate']:.3f})")
    plt.xlabel("Incidence frequency k")
    plt.ylabel("Number of coverage elements")
    plt.title(f"Exponential incidence fit: {subject}")
    plt.legend()
    plt.tight_layout()
    plt.show()


def plot_gamma_incidence_fit(subject: str, incidence_matrix: np.ndarray, save_csv: bool = True):
    dat = _positive_incidence_counts(incidence_matrix)
    x_int = dat["x_int"]
    fit = fit_gamma_incidence_model(incidence_matrix)
    if len(x_int) == 0 or not fit:
        print(f"No positive incidence counts for {subject}")
        return

    vals, counts = np.unique(x_int, return_counts=True)

    fitted = len(x_int) * np.array([
        (
            sp_gamma.cdf(v + 1.0, a=fit["shape"], loc=0.0, scale=fit["scale"])
            - sp_gamma.cdf(v, a=fit["shape"], loc=0.0, scale=fit["scale"])
        ) / max(fit["p_seen"], 1e-12)
        for v in vals
    ], dtype=float)

    plot_df = pd.DataFrame({
        "subject": subject,
        "k": vals,
        "observed_count": counts,
        "fitted_count": fitted,
        "model": "Gamma (incidence-frequency)",
        "fit_input": "incidence_frequency",
    })
    if save_csv:
        _save_subject_fit_plot_csv(plot_df, f"{subject}_gamma_incidence_fit.csv")

    plt.figure(figsize=(6, 5))
    plt.scatter(vals, counts, s=30, label="Observed")
    plt.plot(vals, fitted, label=f"Gamma fit (shape={fit['shape']:.3f}, rate={fit['rate']:.3f})")
    plt.xlabel("Incidence frequency k")
    plt.ylabel("Number of coverage elements")
    plt.title(f"Gamma incidence fit: {subject}")
    plt.legend()
    plt.tight_layout()
    plt.show()


def plot_nb_incidence_fit(subject: str, incidence_matrix: np.ndarray, save_csv: bool = True):
    dat = _positive_incidence_counts(incidence_matrix)
    x_int = dat["x_int"]
    fit = fit_nb_incidence_model(incidence_matrix)
    if len(x_int) == 0 or not fit:
        print(f"No positive incidence counts for {subject}")
        return

    vals, counts = np.unique(x_int, return_counts=True)
    fitted = len(x_int) * np.array([
        sp_nbinom.pmf(v, n=fit["size"], p=fit["prob"]) / max(fit["p_seen"], 1e-12)
        for v in vals
    ], dtype=float)

    plot_df = pd.DataFrame({
        "subject": subject,
        "k": vals,
        "observed_count": counts,
        "fitted_count": fitted,
        "model": "Negative Binomial (incidence-frequency)",
        "fit_input": "incidence_frequency",
    })
    if save_csv:
        _save_subject_fit_plot_csv(plot_df, f"{subject}_negative_binomial_incidence_fit.csv")

    plt.figure(figsize=(6, 5))
    plt.scatter(vals, counts, s=30, label="Observed")
    plt.plot(vals, fitted, label=f"NB fit (size={fit['size']:.3f}, prob={fit['prob']:.3f})")
    plt.xlabel("Incidence frequency k")
    plt.ylabel("Number of coverage elements")
    plt.title(f"Negative Binomial incidence fit: {subject}")
    plt.legend()
    plt.tight_layout()
    plt.show()


def plot_gamma_poisson_fit(subject: str, incidence_matrix: np.ndarray, save_csv: bool = True):
    obs = observed_incidence_sample(incidence_matrix)
    q_pos = np.asarray(obs["q_pos"], dtype=int)
    fit = fit_gamma_poisson_incidence_model(incidence_matrix)
    if len(q_pos) == 0 or not fit:
        print(f"No positive incidence counts for {subject}")
        return

    vals, counts = np.unique(q_pos, return_counts=True)

    def gp_logpmf(k: int, alpha: float, beta: float) -> float:
        return (
            alpha * math.log(beta)
            + gammaln(k + alpha)
            - gammaln(alpha)
            - gammaln(k + 1)
            - (k + alpha) * math.log(beta + 1.0)
        )

    fitted = len(q_pos) * np.array([
        math.exp(gp_logpmf(v, fit["shape"], fit["rate"])) / max(1.0 - fit["p0"], 1e-12)
        for v in vals
    ], dtype=float)

    plot_df = pd.DataFrame({
        "subject": subject,
        "k": vals,
        "observed_count": counts,
        "fitted_count": fitted,
        "model": "Gamma-Poisson (incidence-frequency)",
        "fit_input": "incidence_frequency",
    })
    if save_csv:
        _save_subject_fit_plot_csv(plot_df, f"{subject}_gamma_poisson_incidence_fit.csv")

    plt.figure(figsize=(6, 5))
    plt.scatter(vals, counts, s=30, label="Observed")
    plt.plot(vals, fitted, label=f"Gamma-Poisson fit (alpha={fit['shape']:.3f}, beta={fit['rate']:.3f})")
    plt.xlabel("Incidence frequency k")
    plt.ylabel("Number of coverage elements")
    plt.title(f"Gamma-Poisson incidence fit: {subject}")
    plt.legend()
    plt.tight_layout()
    plt.show()


def plot_zipf_fit(subject: str, counts_vector: np.ndarray, save_csv: bool = True):
    counts = np.asarray(counts_vector, dtype=float)
    counts = counts[np.isfinite(counts)]
    counts = counts[counts > 0]

    if len(counts) == 0:
        print(f"No positive counts for {subject}")
        return

    sorted_counts = np.sort(counts)[::-1]
    ranks = np.arange(1, len(sorted_counts) + 1, dtype=float)
    fit = fit_zipf_mandelbrot_model(sorted_counts, ranks=ranks)
    
    plot_df = pd.DataFrame({
        "subject": subject,
        "rank": ranks,
        "observed_frequency": sorted_counts,
        "fitted_frequency": fit["pred"],
        "model": "Zipf-Mandelbrot (coverage-frequency)",
        "fit_input": "incidence_sum_axis1",
    })
    if save_csv:
        _save_subject_fit_plot_csv(plot_df, f"{subject}_zipf_mandelbrot_fit.csv")

    plt.figure(figsize=(6, 5))
    plt.scatter(ranks, sorted_counts, s=15, label="Observed")
    plt.plot(ranks, fit["pred"], label=f"Zipf fit (s={fit['s']:.3f}, q={fit['q']:.3f})")
   # plt.xscale("log")
   # plt.yscale("log")
    plt.xlabel("Rank")
    plt.ylabel("Frequency")
    plt.title(f"Zipf-Mandelbrot fit: {subject}")
    plt.legend()
    plt.tight_layout()
    plt.show()


def plot_exponential_fit(subject: str, final_hit_count: np.ndarray, save_csv: bool = True):
    x = subject_average_nonzero_hit_counts_from_artifact(final_hit_count)
    fit = fit_exponential_hitcount_model(x)
    if len(x) == 0 or not fit:
        print(f"No positive averaged hit counts for {subject}")
        return

    hist, edges = np.histogram(x, bins="auto")
    mids = 0.5 * (edges[:-1] + edges[1:])
    probs = sp_expon.cdf(edges[1:], loc=0.0, scale=1.0 / fit["rate"]) - sp_expon.cdf(edges[:-1], loc=0.0, scale=1.0 / fit["rate"])
    fitted = len(x) * probs

    plot_df = pd.DataFrame({
        "subject": subject,
        "bin_left": edges[:-1],
        "bin_right": edges[1:],
        "bin_mid": mids,
        "observed_count": hist,
        "fitted_count": fitted,
        "model": "Exponential (hit-count)",
        "fit_input": "nonzero_averaged_hit_count",
    })
    if save_csv:
        _save_subject_fit_plot_csv(plot_df, f"{subject}_exponential_hitcount_fit.csv")

    plt.figure(figsize=(6, 5))
    plt.scatter(mids, hist, s=30, label="Observed")
    plt.plot(mids, fitted, label=f"Exponential fit (rate={fit['rate']:.3f})")
    plt.xlabel("Averaged nonzero hit count")
    plt.ylabel("Binned frequency")
    plt.title(f"Exponential hit-count fit: {subject}")
    plt.legend()
    plt.tight_layout()
    plt.show()


def plot_gamma_fit(subject: str, final_hit_count: np.ndarray, save_csv: bool = True):
    x = subject_average_nonzero_hit_counts_from_artifact(final_hit_count)
    fit = fit_gamma_hitcount_model(x)
    if len(x) == 0 or not fit:
        print(f"No positive averaged hit counts for {subject}")
        return

    hist, edges = np.histogram(x, bins="auto")
    mids = 0.5 * (edges[:-1] + edges[1:])
    probs = sp_gamma.cdf(edges[1:], a=fit["shape"], loc=0.0, scale=fit["scale"]) - sp_gamma.cdf(edges[:-1], a=fit["shape"], loc=0.0, scale=fit["scale"])
    fitted = len(x) * probs

    plot_df = pd.DataFrame({
        "subject": subject,
        "bin_left": edges[:-1],
        "bin_right": edges[1:],
        "bin_mid": mids,
        "observed_count": hist,
        "fitted_count": fitted,
        "model": "Gamma (hit-count)",
        "fit_input": "nonzero_averaged_hit_count",
    })
    if save_csv:
        _save_subject_fit_plot_csv(plot_df, f"{subject}_gamma_hitcount_fit.csv")

    plt.figure(figsize=(6, 5))
    plt.scatter(mids, hist, s=30, label="Observed")
    plt.plot(mids, fitted, label=f"Gamma fit (shape={fit['shape']:.3f}, rate={fit['rate']:.3f})")
    plt.xlabel("Averaged nonzero hit count")
    plt.ylabel("Binned frequency")
    plt.title(f"Gamma hit-count fit: {subject}")
    plt.legend()
    plt.tight_layout()
    plt.show()


def plot_nb_fit(subject: str, final_hit_count: np.ndarray, save_csv: bool = True):
    x = subject_average_nonzero_hit_counts_from_artifact(final_hit_count)
    fit = fit_nb_hitcount_model(x)
    if len(x) == 0 or not fit:
        print(f"No positive averaged hit counts for {subject}")
        return

    x_int = np.round(x).astype(int)
    x_int = x_int[x_int > 0]
    vals, counts = np.unique(x_int, return_counts=True)
    fitted = len(x_int) * sp_nbinom.pmf(vals, n=fit["size"], p=fit["prob"])

    plot_df = pd.DataFrame({
        "subject": subject,
        "k": vals,
        "observed_count": counts,
        "fitted_count": fitted,
        "model": "Negative Binomial (hit-count)",
        "fit_input": "nonzero_averaged_hit_count",
    })
    if save_csv:
        _save_subject_fit_plot_csv(plot_df, f"{subject}_negative_binomial_hitcount_fit.csv")

    plt.figure(figsize=(6, 5))
    plt.scatter(vals, counts, s=30, label="Observed")
    plt.plot(vals, fitted, label=f"NB fit (size={fit['size']:.3f}, prob={fit['prob']:.3f})")
    plt.xlabel("Rounded averaged nonzero hit count")
    plt.ylabel("Frequency")
    plt.title(f"Negative Binomial hit-count fit: {subject}")
    plt.legend()
    plt.tight_layout()
    plt.show()


In [ ]:

# Example plots for all subjects
for subject, payload in sorted(subject_artifacts.items()):
    incidence = np.asarray(payload["incidence"], dtype=np.float32)
    final_hit_count = np.asarray(payload["final_hit_count"], dtype=np.float32)

    # incidence-frequency fits
    plot_poisson_fit(subject, incidence)
    plot_exponential_incidence_fit(subject, incidence)
    plot_gamma_incidence_fit(subject, incidence)
    plot_nb_incidence_fit(subject, incidence)
    plot_gamma_poisson_fit(subject, incidence)

    # zipf on coverage-frequency
    plot_zipf_fit(subject, np.asarray(payload["incidence"], dtype=float).sum(axis=0))

    # hit-count fits
    plot_exponential_fit(subject, final_hit_count)
    plot_gamma_fit(subject, final_hit_count)
    plot_nb_fit(subject, final_hit_count)


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

for subject, payload in sorted(subject_artifacts.items()):
    incidence_mat = np.asarray(payload["incidence"], dtype=float)
    
    col_sums = incidence_mat.sum(axis=0)
    col_sums = col_sums[col_sums > 0]
    
    
    inc_freq = (incidence_mat > 0).sum(axis=0).astype(float)
    inc_freq = inc_freq[inc_freq > 0]
    
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    
    
    sorted_sums = np.sort(col_sums)[::-1]
    axes[0].scatter(np.arange(1, len(sorted_sums)+1), sorted_sums, s=5, alpha=0.5)
    axes[0].set_xscale("log"); axes[0].set_yscale("log")
    axes[0].set_title(f"{subject}: coverage SUM (current)")
    axes[0].set_xlabel("Rank"); axes[0].set_ylabel("Total hits")
    
    
    sorted_freq = np.sort(inc_freq)[::-1]
    axes[1].scatter(np.arange(1, len(sorted_freq)+1), sorted_freq, s=5, alpha=0.5)
    axes[1].set_xscale("log"); axes[1].set_yscale("log")
    axes[1].set_title(f"{subject}: incidence FREQUENCY (option 1)")
    axes[1].set_xlabel("Rank"); axes[1].set_ylabel("# campaigns hit")
    
    plt.tight_layout()
    plt.show()
    
    
    t = incidence_mat.shape[0]
    print(f"{subject}: t={t} campaigns, "
          f"inc_freq range=[{int(inc_freq.min())}, {int(inc_freq.max())}], "
          f"col_sum range=[{col_sums.min():.0f}, {col_sums.max():.0f}]")

In [ ]:
# ============================================================
# Estimators
# ============================================================

def estimate_chao2(incidence_matrix: np.ndarray) -> Dict[str, float]:
    t, F1, F2, q = incidence_frequency_counts(incidence_matrix)
    S_obs = int(np.sum(q > 0))
    if t <= 1:
        return {"estimate": float(S_obs), "S_obs": S_obs, "F1": F1, "F2": F2, "t": t}
    if F2 > 0:
        est = S_obs + ((t - 1) / t) * (F1 * F1) / (2.0 * F2)
    else:
        est = S_obs + ((t - 1) / t) * F1 * (F1 - 1) / 2.0
    return {"estimate": float(est), "S_obs": S_obs, "F1": F1, "F2": F2, "t": t}

def estimate_chao2_bc(incidence_matrix: np.ndarray) -> Dict[str, float]:
    t, F1, F2, q = incidence_frequency_counts(incidence_matrix)
    S_obs = int(np.sum(q > 0))
    if t <= 1:
        return {"estimate": float(S_obs), "S_obs": S_obs, "F1": F1, "F2": F2, "t": t}
    if F2 > 0:
        est = S_obs + ((t - 1) / t) * F1 * (F1 - 1) / (2.0 * (F2 + 1.0))
    else:
        est = S_obs + ((t - 1) / t) * F1 * (F1 - 1) / 2.0
    return {"estimate": float(est), "S_obs": S_obs, "F1": F1, "F2": F2, "t": t}

def estimate_jackknife1(incidence_matrix: np.ndarray) -> Dict[str, float]:
    t, F1, F2, q = incidence_frequency_counts(incidence_matrix)
    S_obs = int(np.sum(q > 0))
    est = S_obs if t <= 1 else S_obs + F1 * (t - 1) / t
    return {"estimate": float(est), "S_obs": S_obs, "F1": F1, "F2": F2, "t": t}

def estimate_chao1(incidence_matrix: np.ndarray) -> Dict[str, float]:
    t, F1, F2, q = incidence_frequency_counts(incidence_matrix)
    S_obs = int(np.sum(q > 0))
    
    if F2 > 0:
        est = float(S_obs + (F1 ** 2) / (2 * F2))
    else:
        est = float(S_obs + (F1 * (F1 - 1)) / 2)
    return {"estimate": float(est), "S_obs": S_obs, "F1": F1, "F2": F2, "t": t}

def estimate_lanumteang_bohning(incidence_matrix: np.ndarray) -> Dict[str, float]:
    t, F1, F2, q = incidence_frequency_counts(incidence_matrix)
    S_obs = int(np.sum(q > 0))
    F3 = int(np.sum(q == 3))
    if F2 == 0:
        return {"estimate": float(S_obs), "S_obs": S_obs, "F1": F1, "F2": F2, "F3": F3, "t": t}
    est = float(S_obs + (3.0 * (F1 ** 3) * F3) / (4.0 * (F2 ** 3)))
    return {"estimate": est, "S_obs": S_obs, "F1": F1, "F2": F2, "F3": F3, "t": t}

def estimate_chiu(incidence_matrix: np.ndarray) -> Dict[str, float]:
    t, F1, F2, q = incidence_frequency_counts(incidence_matrix)
    S_obs = int(np.sum(q > 0))
    F3 = int(np.sum(q == 3))
    F1_p, F2_p, F3_p = max(1, F1), max(1, F2), max(1, F3)
    if F2 > 0:
        f0_hat = F1 / (2.0 * F2)
    else:
        f0_hat = (F1 * (F1 - 1)) / (2.0 * (F2 + 1))
    ratio = (2.0 * F2_p * F2_p) / (3.0 * F1_p * F3_p)
    clipped = min(max(0.5, ratio), 1.0)
    S_hat = S_obs + f0_hat * (2.0 - clipped)
    return {"estimate": float(S_hat), "S_obs": S_obs, "F1": F1, "F2": F2, "F3": F3, "t": t}

def estimate_poisson_from_fit(incidence_matrix: np.ndarray) -> Dict[str, float]:
    fit = fit_poisson_incidence_model(incidence_matrix)
    obs = _positive_incidence_counts(incidence_matrix)
    if not fit:
        return {"estimate": float(obs["S_obs"]), "S_obs": int(obs["S_obs"])}
    return {
        "estimate": float(fit["estimate"]),
        "S_obs": int(fit["S_obs"]),
        "F1": int(obs["F1"]),
        "F2": int(obs["F2"]),
        "t": int(obs["t"]),
        "lambda": float(fit["lambda"]),
        "p0": float(fit["p0"]),
        "p_seen": float(fit["p_seen"]),
        "loglik": float(fit["loglik"]),
        "aic": float(fit["aic"]),
        "bic": float(fit["bic"]),
        "rmse": float(fit["rmse"]),
        "method_note": "Incidence-frequency Poisson estimator.",
    }

def estimate_exponential_from_fit(incidence_matrix: np.ndarray) -> Dict[str, float]:
    fit = fit_exponential_incidence_model(incidence_matrix)
    obs = _positive_incidence_counts(incidence_matrix)
    if not fit:
        return {"estimate": float(obs["S_obs"]), "S_obs": int(obs["S_obs"])}
    return {
        "estimate": float(fit["estimate"]),
        "S_obs": int(fit["S_obs"]),
        "F1": int(obs["F1"]),
        "F2": int(obs["F2"]),
        "t": int(obs["t"]),
        "rate": float(fit["rate"]),
        "scale": float(fit["scale"]),
        "p0": float(fit["p0"]),
        "p_seen": float(fit["p_seen"]),
        "loglik": float(fit["loglik"]),
        "aic": float(fit["aic"]),
        "bic": float(fit["bic"]),
        "rmse": float(fit["rmse"]),
        "method_note": "Incidence-frequency discretized Exponential estimator.",
    }

def estimate_gamma_from_fit(incidence_matrix: np.ndarray) -> Dict[str, float]:
    fit = fit_gamma_incidence_model(incidence_matrix)
    obs = _positive_incidence_counts(incidence_matrix)
    if not fit:
        return {"estimate": float(obs["S_obs"]), "S_obs": int(obs["S_obs"])}
    return {
        "estimate": float(fit["estimate"]),
        "S_obs": int(fit["S_obs"]),
        "F1": int(obs["F1"]),
        "F2": int(obs["F2"]),
        "t": int(obs["t"]),
        "shape": float(fit["shape"]),
        "scale": float(fit["scale"]),
        "rate": float(fit["rate"]),
        "p0": float(fit["p0"]),
        "p_seen": float(fit["p_seen"]),
        "loglik": float(fit["loglik"]),
        "aic": float(fit["aic"]),
        "bic": float(fit["bic"]),
        "rmse": float(fit["rmse"]),
        "method_note": "Incidence-frequency discretized Gamma estimator.",
    }

def estimate_nb_from_fit(incidence_matrix: np.ndarray) -> Dict[str, float]:
    fit = fit_nb_incidence_model(incidence_matrix)
    obs = _positive_incidence_counts(incidence_matrix)
    if not fit:
        return {"estimate": float(obs["S_obs"]), "S_obs": int(obs["S_obs"])}
    return {
        "estimate": float(fit["estimate"]),
        "S_obs": int(fit["S_obs"]),
        "F1": int(obs["F1"]),
        "F2": int(obs["F2"]),
        "t": int(obs["t"]),
        "size": float(fit["size"]),
        "prob": float(fit["prob"]),
        "mean": float(fit["mean"]),
        "p0": float(fit["p0"]),
        "p_seen": float(fit["p_seen"]),
        "loglik": float(fit["loglik"]),
        "aic": float(fit["aic"]),
        "bic": float(fit["bic"]),
        "rmse": float(fit["rmse"]),
        "method_note": "Incidence-frequency Negative Binomial estimator.",
    }




def estimate_gamma_poisson_from_fit(incidence_matrix: np.ndarray) -> Dict[str, float]:
    fit = fit_gamma_poisson_incidence_model(incidence_matrix)
    obs = observed_incidence_sample(incidence_matrix)
    if not fit:
        return {"estimate": float(obs["S_obs"]), "S_obs": int(obs["S_obs"])}
    return {
        "estimate": float(fit["estimate"]),
        "S_obs": int(fit["S_obs"]),
        "F1": int(obs["F1"]),
        "F2": int(obs["F2"]),
        "t": int(obs["t"]),
        "shape": float(fit["shape"]),
        "rate": float(fit["rate"]),
        "scale": float(fit["scale"]),
        "p0": float(fit["p0"]),
        "p_seen": float(fit["p_seen"]),
        "loglik": float(fit["loglik"]),
        "aic": float(fit["aic"]),
        "bic": float(fit["bic"]),
        "rmse": float(fit["rmse"]),
        "method_note": "Gamma-Poisson estimator from incidence-frequency fit.",
    }

def find_S_threshold(J, s, q, max_search=100000):
    for S in range(1, max_search):
        ranks = np.arange(1, S + 1)
        weights = (ranks + q) ** (-s)
        C = 1.0 / np.sum(weights)
        n_S = J * C * (S + q) ** (-s)
        if n_S < 1:
            return S - 1
    return None


def estimate_zipf_mandelbrot(incidence_matrix: np.ndarray) -> Dict[str, float]:
    # sum across intervals (axis=0) to get per-block (per-element) incidence counts
    counts = np.asarray(incidence_matrix, dtype=float).sum(axis=0)
    counts = counts[np.isfinite(counts)]
    counts = counts[counts > 0]
    if len(counts) == 0:
        return {"estimate": 0.0, "S_obs": 0}

    S_obs = int(len(counts))
    fit = fit_zipf_mandelbrot_model(counts)
    if not fit:
        return {"estimate": float(S_obs), "S_obs": S_obs}

    S_hat = find_S_threshold(fit["J"], fit["s"], fit["q"])
    if S_hat is None:
        S_hat = S_obs

    return {
        "estimate": float(S_hat),
        "S_obs": S_obs,
        "J": float(fit["J"]),
        "s": float(fit["s"]),
        "q": float(fit["q"]),
        "loglik": float(fit["loglik"]),
        "aic": float(fit["aic"]),
        "bic": float(fit["bic"]),
        "rmse": float(fit["rmse"]),
        "method_note": "Zipf-Mandelbrot estimator on coverage-frequency rank data.",
    }


def estimate_beta_binomial_from_gp_fit(incidence_matrix: np.ndarray) -> Dict[str, float]:
    """
    Beta-Binomial species richness estimator (Chiu, 2022, Eq. 7).
    """
    gp_fit = fit_gamma_poisson_incidence_model(incidence_matrix)
    obs = observed_incidence_sample(incidence_matrix)

    S_obs = int(obs["S_obs"])
    t = int(obs["t"])
    F1 = int(obs["F1"])
    F2 = int(obs["F2"])
    q = obs["q"]
    F3 = int(np.sum(np.asarray(q) == 3))

    if not gp_fit or t < 1:
        return {"estimate": float(S_obs), "S_obs": S_obs, "F1": F1, "F2": F2, "F3": F3, "t": t}

    alpha_gp = float(gp_fit["shape"])
    beta_gp  = float(gp_fit["rate"])   # rate parameter

   
    alpha_bb = alpha_gp
    beta_bb  = beta_gp * t

    try:
        log_p0_bb = (
            gammaln(alpha_bb + beta_bb) + gammaln(beta_bb + t)
            - gammaln(beta_bb) - gammaln(alpha_bb + beta_bb + t)
        )
        p0_bb = float(math.exp(log_p0_bb))
    except Exception:
        p0_bb = float(gp_fit.get("p0", 0.0))

    p0_bb = float(np.clip(p0_bb, 0.0, 1.0 - 1e-12))


    if F2 > 0:
        Q0_Chao2 = float((t - 1) / t * F1 ** 2 / (2.0 * F2))
    else:
        Q0_Chao2 = float((t - 1) / t * F1 * (F1 - 1) / 2.0)

    # F: bias-correction factor using tripletons
    Q1_eff = max(F1, 1)
    Q2_eff = max(F2, 1)
    Q3_eff = max(F3, 1)
    ratio = (2.0 * Q2_eff ** 2) / (3.0 * Q1_eff * Q3_eff)
    F_val = float(max(0.5, min(ratio, 1.0)))

    S_hat = float(S_obs + Q0_Chao2 * (2.0 - F_val))

    return {
        "estimate": S_hat,
        "S_obs": S_obs,
        "F1": F1,
        "F2": F2,
        "F3": F3,
        "t": t,
        "alpha_bb": alpha_bb,
        "beta_bb": beta_bb,
        "alpha_gp": alpha_gp,
        "beta_gp": beta_gp,
        "p0": p0_bb,
        "p_seen": float(1.0 - p0_bb),
        "Q0_Chao2": Q0_Chao2,
        "F_val": F_val,
        "method_note": (
            "Beta-Binomial estimator (Chiu 2022 Eq. 7). "
            "Parameters alpha_BB=alpha_GP, beta_BB=beta_GP*T derived from Gamma-Poisson fit."
        ),
    }


In [ ]:

# ============================================================
# Estimator comparison
# ============================================================

def canonical_subject_name(subject_name: str) -> str:
    subject_name = str(subject_name)
    subject_name = re.sub(r"_aflpp_run_\d+$", "", subject_name)
    subject_name = re.sub(r"_run_\d+$", "", subject_name)
    return subject_name


def explicit_estimator_label(estimator_name: str, estimator_input: str) -> str:
    base = str(estimator_name).strip()
    label_map = {
        ("Poisson", "incidence_frequency"): "Poisson (incidence-frequency estimator)",
        ("Exponential", "incidence_frequency"): "Exponential (incidence-frequency estimator)",
        ("Gamma", "incidence_frequency"): "Gamma (incidence-frequency estimator)",
        ("Negative Binomial", "incidence_frequency"): "Negative Binomial (incidence-frequency estimator)",
        ("Gamma-Poisson", "incidence_frequency"): "Gamma-Poisson (incidence-frequency estimator)",
        ("ztPoisson", "incidence_frequency"): "ztPoisson (incidence-frequency estimator)",
        ("ztNB", "incidence_frequency"): "ztNB (incidence-frequency estimator)",
        ("Zipf-Mandelbrot", "incidence_sum_axis1"): "Zipf-Mandelbrot (coverage-frequency estimator)",
        ("Chao2", "incidence_frequency"): "Chao2 (incidence-based)",
        ("Chao2_bc", "incidence_frequency"): "Chao2_bc (incidence-based)",
        ("Jackknife1", "incidence_frequency"): "Jackknife1 (incidence-based)",
        ("Chao1", "incidence_frequency"): "Chao1 (incidence-based)",
        ("Lanumteang-Bohning", "incidence_frequency"): "Lanumteang-Bohning (incidence-based)",
        ("Chiu", "incidence_frequency"): "Chiu (incidence-based)",
        ("Beta-Binomial", "incidence_frequency"): "Beta-Binomial (incidence-based)",
    }
    return label_map.get((base, estimator_input), f"{base} ({estimator_input})")


STANDARD_PARAMETRIC_ESTIMATORS = [
    "Poisson",
    "Exponential",
    "Gamma",
    "Negative Binomial",
    "Gamma-Poisson",
    "Zipf-Mandelbrot",
]
NON_PARAMETRIC_ESTIMATORS = ["Chao2", "Chao2_bc", "Jackknife1", "Chao1", "Lanumteang-Bohning", "Chiu", "Beta-Binomial"]
ZT_PARAMETRIC_ESTIMATORS = ["ztPoisson", "ztNB"]
HITCOUNT_PARAMETRIC_ESTIMATORS = ["Poisson", "Exponential", "Gamma", "Negative Binomial"]
PARAMETRIC_PLUS_ZT_ESTIMATORS = STANDARD_PARAMETRIC_ESTIMATORS + ZT_PARAMETRIC_ESTIMATORS


def estimator_family(estimator_name: str, estimator_input: str) -> str:
    if estimator_input == "hit_count":
        return "hit_count_parametric"
    if estimator_name in STANDARD_PARAMETRIC_ESTIMATORS:
        return "standard_parametric"
    if estimator_name in NON_PARAMETRIC_ESTIMATORS:
        return "non_parametric"
    if estimator_name in ZT_PARAMETRIC_ESTIMATORS:
        return "zero_truncated_parametric"
    return "other"


comparison_rows = []

for artifact_key, subject_payload in sorted(subject_artifacts.items()):
    metadata = subject_payload.get("metadata", {})
    subject = canonical_subject_name(metadata.get("subject", artifact_key))

    incidence = np.asarray(subject_payload["incidence"], dtype=np.float32)
    final_hit_count = np.asarray(subject_payload["final_hit_count"], dtype=np.float32)

    obs_inc = observed_incidence_sample(incidence)
    S_obs_inc = int(obs_inc["S_obs"])
    S_obs_hits = observed_species_from_counts(final_hit_count)

    has_known_total = subject in KNOWN_TOTALS
    known_total = float(KNOWN_TOTALS[subject]) if has_known_total else np.nan

    estimator_functions = [
        ("Chao2", "incidence_frequency", lambda: estimate_chao2(incidence)),
        ("Chao2_bc", "incidence_frequency", lambda: estimate_chao2_bc(incidence)),
        ("Jackknife1", "incidence_frequency", lambda: estimate_jackknife1(incidence)),
        ("Chao1", "incidence_frequency", lambda: estimate_chao1(incidence)),
        ("Poisson", "incidence_frequency", lambda: estimate_poisson_from_fit(incidence)),
        ("Exponential", "incidence_frequency", lambda: estimate_exponential_from_fit(incidence)),
        ("Gamma", "incidence_frequency", lambda: estimate_gamma_from_fit(incidence)),
        ("Negative Binomial", "incidence_frequency", lambda: estimate_nb_from_fit(incidence)),
        ("Gamma-Poisson", "incidence_frequency", lambda: estimate_gamma_poisson_from_fit(incidence)),
        ("Lanumteang-Bohning", "incidence_frequency", lambda: estimate_lanumteang_bohning(incidence)),
        ("Chiu", "incidence_frequency", lambda: estimate_chiu(incidence)),
        ("Beta-Binomial", "incidence_frequency", lambda: estimate_beta_binomial_from_gp_fit(incidence)),
        ("Zipf-Mandelbrot", "incidence_sum_axis1", lambda: estimate_zipf_mandelbrot(incidence)),
    ]

    for est_name, estimator_input, fn in estimator_functions:
        try:
            res = fn()
        except Exception as e:
            print(f"[WARN] {subject} - {est_name} ({estimator_input}) failed: {e}")
            res = {}

        if not res:
            continue

        est_total = float(res.get("estimate", np.nan))
        observed_total_for_error = S_obs_hits if estimator_input == "hit_count" else S_obs_inc
        signed_err = est_total - known_total if has_known_total and np.isfinite(est_total) else np.nan
        abs_err = abs(signed_err) if np.isfinite(signed_err) else np.nan
        rel_err = abs_err / known_total if has_known_total and known_total > 0 and np.isfinite(abs_err) else np.nan
        coverage_completion = observed_total_for_error / known_total if has_known_total and known_total > 0 else np.nan

        comparison_rows.append({
            "subject": subject,
            "artifact_key": artifact_key,
            "program_type": "synthetic" if has_known_total else "real_world",
            "has_known_total": has_known_total,
            "target_type": "known_total" if has_known_total else "unknown",
            "known_total": known_total,
            "observed_total": S_obs_inc,
            "observed_total_hitcount": S_obs_hits,
            "observed_total_for_estimator": observed_total_for_error,
            "estimate_total": est_total,
            "estimated_unseen_vs_observed": float(max(est_total - observed_total_for_error, 0.0)) if np.isfinite(est_total) else np.nan,
            "signed_error": signed_err,
            "abs_error": abs_err,
            "rel_error": rel_err,
            "coverage_completion": coverage_completion,
            "estimator": est_name,
            "estimator_input": estimator_input,
            "estimator_label": explicit_estimator_label(est_name, estimator_input),
            "n_campaigns": int(metadata.get("n_campaigns", 0)),
            "F1": res.get("F1", np.nan),
            "F2": res.get("F2", np.nan),
            "F3": res.get("F3", np.nan),
            "t": res.get("t", np.nan),
            "p0": res.get("p0", np.nan),
            "p_seen": res.get("p_seen", np.nan),
            "loglik": res.get("loglik", np.nan),
            "aic": res.get("aic", np.nan),
            "bic": res.get("bic", np.nan),
            "rmse": res.get("rmse", np.nan),
            "lambda": res.get("lambda", np.nan),
            "shape": res.get("shape", np.nan),
            "scale": res.get("scale", np.nan),
            "rate": res.get("rate", np.nan),
            "size": res.get("size", np.nan),
            "prob": res.get("prob", np.nan),
            "mean": res.get("mean", np.nan),
            "J": res.get("J", np.nan),
            "s": res.get("s", np.nan),
            "q": res.get("q", np.nan),
            "method_note": res.get("method_note", ""),
            "estimator_family": estimator_family(est_name, estimator_input),
        })

comparison_df = pd.DataFrame(comparison_rows).sort_values(["subject", "estimator_input", "estimator"]).reset_index(drop=True)

comparison_known_df = comparison_df[comparison_df["has_known_total"]].copy()
comparison_unknown_df = comparison_df[~comparison_df["has_known_total"]].copy()

display(comparison_df.head(30))


In [ ]:

# ============================================================
# Summary tables
# ============================================================

def summarise_estimator_performance(df: pd.DataFrame, group_cols: List[str]) -> pd.DataFrame:
    if df.empty:
        return pd.DataFrame()
    return (
        df.groupby(group_cols, dropna=False)
          .agg(
              n=("estimate_total", "size"),
              mean_estimate=("estimate_total", "mean"),
              mean_abs_error=("abs_error", "mean"),
              median_abs_error=("abs_error", "median"),
              rmse=("signed_error", lambda s: float(np.sqrt(np.mean(np.square(s.dropna())))) if s.notna().any() else np.nan),
              mean_rel_error=("rel_error", "mean"),
              median_rel_error=("rel_error", "median"),
              mean_signed_error=("signed_error", "mean"),
              mean_completion=("coverage_completion", "mean"),
          )
          .reset_index()
          .sort_values(group_cols)
    )

def summarise_relative_behavior(df: pd.DataFrame, group_cols: List[str]) -> pd.DataFrame:
    if df.empty:
        return pd.DataFrame()
    return (
        df.groupby(group_cols, dropna=False)
          .agg(
              n=("estimate_total", "size"),
              mean_estimate=("estimate_total", "mean"),
              median_estimate=("estimate_total", "median"),
              std_estimate=("estimate_total", "std"),
              min_estimate=("estimate_total", "min"),
              max_estimate=("estimate_total", "max"),
              mean_unseen_vs_observed=("estimated_unseen_vs_observed", "mean"),
          )
          .reset_index()
          .sort_values(group_cols)
    )

def subset_by_estimators(df: pd.DataFrame, estimator_labels: List[str]) -> pd.DataFrame:
    return df[df["estimator_label"].isin(estimator_labels)].copy()

INCIDENCE_PARAMETRIC_LABELS = [
    "Poisson (incidence-frequency estimator)",
    "Exponential (incidence-frequency estimator)",
    "Gamma (incidence-frequency estimator)",
    "Negative Binomial (incidence-frequency estimator)",
    "Gamma-Poisson (incidence-frequency estimator)",
    "Zipf-Mandelbrot (coverage-frequency estimator)",
]
NON_PARAMETRIC_LABELS = [
    "Chao2 (incidence-based)",
    "Chao2_bc (incidence-based)",
    "Jackknife1 (incidence-based)",
    "Chao1 (incidence-based)",
    "Lanumteang-Bohning (incidence-based)",
    "Chiu (incidence-based)",
    "Beta-Binomial (incidence-based)",
]



estimator_summary_overall_df = summarise_estimator_performance(comparison_known_df, ["estimator_label"])
estimator_summary_by_subject_df = summarise_estimator_performance(comparison_known_df, ["subject", "estimator_label"])
relative_behavior_overall_df = summarise_relative_behavior(comparison_unknown_df, ["estimator_label"])
relative_behavior_by_subject_df = summarise_relative_behavior(comparison_unknown_df, ["subject", "estimator_label"])

summary_groups = {
    "standard_parametric_incidence": INCIDENCE_PARAMETRIC_LABELS,
    "non_parametric_incidence": NON_PARAMETRIC_LABELS,
}

grouped_known_summaries = {}
grouped_unknown_summaries = {}
grouped_known_subject_summaries = {}
grouped_unknown_subject_summaries = {}

for group_name, estimator_list in summary_groups.items():
    known_sub = subset_by_estimators(comparison_known_df, estimator_list)
    unknown_sub = subset_by_estimators(comparison_unknown_df, estimator_list)

    grouped_known_summaries[group_name] = summarise_estimator_performance(known_sub, ["estimator_label"])
    grouped_known_subject_summaries[group_name] = summarise_estimator_performance(known_sub, ["subject", "estimator_label"])
    grouped_unknown_summaries[group_name] = summarise_relative_behavior(unknown_sub, ["estimator_label"])
    grouped_unknown_subject_summaries[group_name] = summarise_relative_behavior(unknown_sub, ["subject", "estimator_label"])

best_by_subject_df = pd.DataFrame()
best_overall_df = pd.DataFrame()
if not estimator_summary_by_subject_df.empty:
    best_by_subject_df = estimator_summary_by_subject_df.loc[
        estimator_summary_by_subject_df.groupby("subject")["mean_abs_error"].idxmin()
    ].sort_values("subject").reset_index(drop=True)

if not estimator_summary_overall_df.empty:
    best_overall_df = estimator_summary_overall_df.sort_values(["mean_abs_error", "rmse", "mean_rel_error"]).reset_index(drop=True)

display(estimator_summary_overall_df)
display(grouped_known_summaries["standard_parametric_incidence"])
display(grouped_known_summaries["non_parametric_incidence"])



In [ ]:

# ============================================================
# Save tables
# ============================================================

comparison_df.to_csv(SUMMARIES_DIR / "estimator_comparison_subject_artifacts.csv", index=False)
comparison_known_df.to_csv(SUMMARIES_DIR / "estimator_comparison_known_totals_only.csv", index=False)
comparison_unknown_df.to_csv(SUMMARIES_DIR / "estimator_comparison_unknown_totals_only.csv", index=False)

estimator_summary_overall_df.to_csv(SUMMARIES_DIR / "estimator_summary_overall_known_totals.csv", index=False)
estimator_summary_by_subject_df.to_csv(SUMMARIES_DIR / "estimator_summary_by_subject_known_totals.csv", index=False)
relative_behavior_overall_df.to_csv(SUMMARIES_DIR / "estimator_relative_behavior_overall_unknown_totals.csv", index=False)
relative_behavior_by_subject_df.to_csv(SUMMARIES_DIR / "estimator_relative_behavior_by_subject_unknown_totals.csv", index=False)
best_by_subject_df.to_csv(SUMMARIES_DIR / "best_estimator_by_subject_known_totals.csv", index=False)
best_overall_df.to_csv(SUMMARIES_DIR / "best_estimator_overall_known_totals.csv", index=False)

for group_name, df_out in grouped_known_summaries.items():
    df_out.to_csv(SUMMARIES_DIR / f"estimator_summary_{group_name}_known_totals.csv", index=False)
for group_name, df_out in grouped_known_subject_summaries.items():
    df_out.to_csv(SUMMARIES_DIR / f"estimator_summary_{group_name}_by_subject_known_totals.csv", index=False)
for group_name, df_out in grouped_unknown_summaries.items():
    df_out.to_csv(SUMMARIES_DIR / f"estimator_relative_behavior_{group_name}_unknown_totals.csv", index=False)
for group_name, df_out in grouped_unknown_subject_summaries.items():
    df_out.to_csv(SUMMARIES_DIR / f"estimator_relative_behavior_{group_name}_by_subject_unknown_totals.csv", index=False)

print("Saved summary tables to:", SUMMARIES_DIR)


In [ ]:

# ============================================================
# Extra incidence-frequency / hit-count / zipf summary exports
# ============================================================

incidence_fit_df = subject_fit_df[
    subject_fit_df["fit_input"] == "incidence_frequency"
].copy()

hitcount_fit_df = subject_fit_df[
    subject_fit_df["fit_input"] == "nonzero_averaged_hit_count"
].copy()

incidence_estimator_df = comparison_df[
    comparison_df["estimator_input"] == "incidence_frequency"
].copy()


zipf_estimator_df = comparison_df[
    comparison_df["estimator_input"] == "incidence_sum_axis1"
].copy()

# all-subject exports by estimator input
incidence_estimator_df.to_csv(
    SUMMARIES_DIR / "incidence_frequency_estimator_comparison_all_subjects.csv",
    index=False
)


zipf_estimator_df.to_csv(
    SUMMARIES_DIR / "coverage_frequency_estimator_comparison_all_subjects.csv",
    index=False
)

# synthetic / real-world splits for incidence estimators
for program_type, safe_name in [("synthetic", "synthetic_programs"), ("real_world", "real_world_programs")]:
    incidence_estimator_df[incidence_estimator_df["program_type"] == program_type].to_csv(
        SUMMARIES_DIR / f"incidence_frequency_estimator_comparison_{safe_name}.csv",
        index=False,
    )

best_incidence_aic_df = (
    incidence_fit_df.sort_values(["subject", "aic"])
    .groupby("subject", as_index=False)
    .first()
)
best_incidence_aic_df.to_csv(
    SUMMARIES_DIR / "best_incidence_frequency_model_by_aic.csv",
    index=False
)

best_incidence_rmse_df = (
    incidence_fit_df.sort_values(["subject", "rmse"])
    .groupby("subject", as_index=False)
    .first()
)
best_incidence_rmse_df.to_csv(
    SUMMARIES_DIR / "best_incidence_frequency_model_by_rmse.csv",
    index=False
)

best_hitcount_aic_df = (
    hitcount_fit_df.sort_values(["subject", "aic"])
    .groupby("subject", as_index=False)
    .first()
)
best_hitcount_aic_df.to_csv(
    SUMMARIES_DIR / "best_hit_count_model_by_aic.csv",
    index=False
)

best_hitcount_rmse_df = (
    hitcount_fit_df.sort_values(["subject", "rmse"])
    .groupby("subject", as_index=False)
    .first()
)
best_hitcount_rmse_df.to_csv(
    SUMMARIES_DIR / "best_hit_count_model_by_rmse.csv",
    index=False
)

zipf_df = subject_fit_df[
    subject_fit_df["fit_input"] == "incidence_sum_axis1"
].copy()

zipf_cols = [c for c in [
    "subject", "model", "fit_family", "n_obs", "J", "s", "q", "loglik", "aic", "bic", "rmse"
] if c in zipf_df.columns]

zipf_df[zipf_cols].to_csv(
    SUMMARIES_DIR / "zipf_mandelbrot_coverage_frequency_summary.csv",
    index=False
)

print("Saved incidence estimator CSVs, hit-count estimator CSVs, and synthetic/real-world splits to:", SUMMARIES_DIR)


In [ ]:

# ============================================================
# Plot configuration
# ============================================================

ESTIMATOR_ORDER = [
    "Chao2 (incidence-based)",
    "Chao2_bc (incidence-based)",
    "Jackknife1 (incidence-based)",
    "Chao1 (incidence-based)",
    "Poisson (incidence-frequency estimator)",
    "Exponential (incidence-frequency estimator)",
    "Gamma (incidence-frequency estimator)",
    "Negative Binomial (incidence-frequency estimator)",
    "Gamma-Poisson (incidence-frequency estimator)",
    "Lanumteang-Bohning (incidence-based)",
    "Chiu (incidence-based)",
    "Beta-Binomial (incidence-based)",
    "Zipf-Mandelbrot (coverage-frequency estimator)",
    
]

df_known = comparison_df[comparison_df["has_known_total"]].copy()
df_known = df_known[df_known["estimator_label"].isin(ESTIMATOR_ORDER)].copy()
df_known["estimator_label"] = pd.Categorical(df_known["estimator_label"], categories=ESTIMATOR_ORDER, ordered=True)

df_unknown = comparison_df[~comparison_df["has_known_total"]].copy()
df_unknown = df_unknown[df_unknown["estimator_label"].isin(ESTIMATOR_ORDER)].copy()
df_unknown["estimator_label"] = pd.Categorical(df_unknown["estimator_label"], categories=ESTIMATOR_ORDER, ordered=True)


In [ ]:
# ============================================================
# Controlled benchmarks: absolute error, relative error, RMSE
# ============================================================

def plot_bar_with_subject_points(df, y_col: str, title: str, ylabel: str, csv_prefix: str):
    raw_cols = [
        "subject", "program_type", "has_known_total", "known_total",
        "estimator_label", "estimate_total", "signed_error", "abs_error", "rel_error", y_col,
    ]
    _save_plot_csv(df.sort_values(["estimator_label", "subject"]), f"{csv_prefix}_raw.csv", raw_cols)

    summary_df = (
        df.groupby("estimator_label", observed=False)[y_col]
        .agg(["mean", "std", "count"])
        .reset_index()
        .rename(columns={"mean": f"{y_col}_mean", "std": f"{y_col}_sd", "count": "n_subjects"})
    )
    summary_df["estimator_label"] = pd.Categorical(summary_df["estimator_label"], categories=ESTIMATOR_ORDER, ordered=True)
    summary_df = summary_df.sort_values("estimator_label")
    _save_plot_csv(summary_df, f"{csv_prefix}_summary.csv")




rmse_input_df = (
    df_known[["subject", "estimator_label", "signed_error", "abs_error", "rel_error", "estimate_total", "known_total"]]
    .sort_values(["estimator_label", "subject"])
    .copy()
)
_save_plot_csv(rmse_input_df, "rmse_comparison_input.csv")

rmse_df = (
    df_known.groupby("estimator_label", observed=False)["signed_error"]
    .apply(lambda s: np.sqrt(np.mean(np.square(s.dropna()))) if s.notna().any() else np.nan)
    .reset_index(name="RMSE")
)
rmse_df["estimator_label"] = pd.Categorical(rmse_df["estimator_label"], categories=ESTIMATOR_ORDER, ordered=True)
rmse_df = rmse_df.sort_values("estimator_label")
_save_plot_csv(rmse_df, "rmse_comparison_summary.csv")



In [ ]:
# ============================================================
# Controlled benchmarks: per-subject estimator comparison
# ============================================================

_cols = [
    "subject", "estimator_label", "estimate_total",
    "known_total", "observed_total", "abs_error", "rel_error", "signed_error",
    "n_campaigns", "n_rows_total", "n_blocks_total",
]
_available_cols = [c for c in _cols if c in df_known.columns]

df_known_plot_data = (
    df_known[_available_cols]
    .sort_values(["subject", "estimator_label"])
    .copy()
)

std_dev = (
    df_known_plot_data.groupby("estimator_label")["estimate_total"]
    .std(ddof=1)
    .round()
    .astype(int)
    .rename("std_dev")
)
df_known_plot_data = df_known_plot_data.merge(std_dev, on="estimator_label", how="left")
_save_plot_csv(df_known_plot_data, "synthetic_estimator_comparison_per_subject.csv")

truth_df = df_known[["subject", "known_total"]].drop_duplicates().sort_values("subject")



In [ ]:
# ============================================================
# Real-world programs: per-subject estimator comparison
# ============================================================

_cols = [
    "subject", "estimator_label", "estimate_total",
    
    "n_campaigns", "n_rows_total", "n_blocks_total",
]
_available_cols = [c for c in _cols if c in df_unknown.columns]

df_unknown_pointplot = (
    df_unknown[_available_cols]
    .sort_values(["subject", "estimator_label"])
    .copy()
)
_save_plot_csv(df_unknown_pointplot, "real_world_estimator_comparison_per_subject.csv")




In [ ]:
# ============================================================
# Real-world programs: behavior across subjects
# ============================================================

lineplot_df = (
    df_unknown[["subject", "estimator_label", "estimate_total"]]
    .sort_values(["subject", "estimator_label"])
    .copy()
)
_save_plot_csv(lineplot_df, "estimator_behavior_across_subjects.csv")



df_unknown = df_unknown.copy()
df_unknown["mean_per_subject"] = df_unknown.groupby("subject")["estimate_total"].transform("mean")
df_unknown["relative_diff"] = df_unknown["estimate_total"] - df_unknown["mean_per_subject"]

boxplot_raw_df = (
    df_unknown[["subject", "estimator_label", "estimate_total", "mean_per_subject", "relative_diff"]]
    .sort_values(["estimator_label", "subject"])
    .copy()
)
_save_plot_csv(boxplot_raw_df, "deviation_from_subject_mean_raw.csv")

boxplot_summary_df = (
    df_unknown.groupby("estimator_label", observed=False)["relative_diff"]
    .agg(
        n="count",
        mean="mean",
        std="std",
        min="min",
        q1=lambda s: s.quantile(0.25),
        median="median",
        q3=lambda s: s.quantile(0.75),
        max="max",
    )
    .reset_index()
)
boxplot_summary_df["estimator_label"] = pd.Categorical(boxplot_summary_df["estimator_label"], categories=ESTIMATOR_ORDER, ordered=True)
boxplot_summary_df = boxplot_summary_df.sort_values("estimator_label")
_save_plot_csv(boxplot_summary_df, "deviation_from_subject_mean_summary.csv")




In [ ]:

MODEL_ORDER_GENERIC = [
    "Poisson (incidence-frequency)",
    "Exponential (incidence-frequency)",
    "Gamma (incidence-frequency)",
    "Negative Binomial (incidence-frequency)",
    "Gamma-Poisson (incidence-frequency)",
    "Zipf-Mandelbrot (coverage-frequency)",
    "ztPoisson (incidence-frequency)",
    "ztNB (incidence-frequency)",
]

ESTIMATOR_ORDER_GENERIC = [
    "Chao2 (incidence-based)",
    "Chao2_bc (incidence-based)",
    "Jackknife1 (incidence-based)",
    "Chao1 (incidence-based)",
    "Poisson (incidence-frequency estimator)",
    "Exponential (incidence-frequency estimator)",
    "Gamma (incidence-frequency estimator)",
    "Negative Binomial (incidence-frequency estimator)",
    "Gamma-Poisson (incidence-frequency estimator)",
    "Lanumteang-Bohning (incidence-based)",
    "Chiu (incidence-based)",
    "Beta-Binomial (incidence-based)",
    "Zipf-Mandelbrot (coverage-frequency estimator)",
    "ztPoisson (incidence-frequency estimator)",
    "ztNB (incidence-frequency estimator)",
]





In [ ]:

import pandas as pd
import numpy as np


df = comparison_known_df.copy()
df = df[df["subject"].isin(KNOWN_TOTALS.keys())].copy()


df_subject = (
    df.groupby(["subject", "estimator", "estimator_input", "estimator_label"], as_index=False)
      .agg(estimate_total=("estimate_total", "mean"))
)


df_subject["known_total"] = df_subject["subject"].map(KNOWN_TOTALS)
df_subject["signed_error"] = df_subject["estimate_total"] - df_subject["known_total"]
df_subject["abs_error"] = df_subject["signed_error"].abs()
df_subject["rel_error"] = df_subject["abs_error"] / df_subject["known_total"]


df_subject = df_subject.sort_values(["subject", "estimator_input", "estimator_label"]).reset_index(drop=True)
df_subject.to_csv(SUMMARIES_DIR / "subject_level_estimator_comparison.csv", index=False)


group_views = {
    "standard_parametric_incidence": (
        (df_subject["estimator_input"] == "incidence_frequency") &
        (df_subject["estimator"].isin(STANDARD_PARAMETRIC_ESTIMATORS))
    ),
    "non_parametric_incidence": (
        (df_subject["estimator_input"] == "incidence_frequency") &
        (df_subject["estimator"].isin(NON_PARAMETRIC_ESTIMATORS))
    ),
    "zt_only": df_subject["estimator"].isin(ZT_PARAMETRIC_ESTIMATORS),
    "hit_count_parametric": df_subject["estimator_input"] == "hit_count",
    "parametric_plus_hitcount": (
        ((df_subject["estimator_input"] == "incidence_frequency") & (df_subject["estimator"].isin(STANDARD_PARAMETRIC_ESTIMATORS))) |
        (df_subject["estimator_input"] == "hit_count")
    ),
}

for view_name, mask in group_views.items():
    out = df_subject[mask].copy()
    out.to_csv(SUMMARIES_DIR / f"subject_level_estimator_comparison_{view_name}.csv", index=False)

df_subject.head()


In [ ]:
import pandas as pd
from pathlib import Path


INPUT_CSV = SUMMARIES_DIR / "subject_parametric_fit_summary.csv"
OUTPUT_DIR = Path(".")

df = pd.read_csv(INPUT_CSV)


df["subject"] = df["subject"].astype(str)

is_synthetic = df["subject"].str.startswith("parser")
is_real = ~is_synthetic


fit_family = df["fit_family"].fillna("").astype(str).str.lower()
fit_input = df["fit_input"].fillna("").astype(str).str.lower()


is_zt = fit_family.str.contains("zero-truncated", case=False, na=False)

is_hitcount = fit_input.eq("nonzero_averaged_hit_count")

is_incidence = fit_input.isin(["incidence_frequency", "incidence_sum_axis1"]) & (~is_zt)


outputs = {
    "model_fit_incidence_real.csv": df[is_incidence & is_real].copy(),
    "model_fit_incidence_synthetic.csv": df[is_incidence & is_synthetic].copy(),
    "model_fit_zt_real.csv": df[is_zt & is_real].copy(),
    "model_fit_zt_synthetic.csv": df[is_zt & is_synthetic].copy(),
}


OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

for filename, subdf in outputs.items():
    out_path = OUTPUT_DIR / filename
    subdf.to_csv(out_path, index=False)
    print(f"Saved: {out_path} ({len(subdf)} rows)")


print("\nSummary:")
for filename, subdf in outputs.items():
    subjects = sorted(subdf["subject"].unique()) if not subdf.empty else []
    models = sorted(subdf["model"].astype(str).unique()) if not subdf.empty else []
    print(f"\n{filename}")
    print(f"  rows    : {len(subdf)}")
    print(f"  subjects: {subjects}")
    print(f"  models  : {models}")

In [ ]:
import pandas as pd

df = pd.read_csv(SUMMARIES_DIR / "estimator_comparison_known_totals_only.csv")

# Group by estimator
df_summary = (
    df.groupby("estimator")["abs_error"]
    .agg(MAE="mean", SD="std")
    .reset_index()
)

# Save to CSV
df_summary.to_csv(SUMMARIES_DIR / "estimator_mae_sd_summary.csv", index=False)

print(df_summary)

In [ ]:
import pandas as pd

# Load the correct CSV
df = pd.read_csv(SUMMARIES_DIR / "estimator_comparison_known_totals_only.csv")

# Compute bias and variance per estimator
bias_variance_df = (
    df.groupby("estimator")["signed_error"]
      .agg(
          bias="mean",
          variance="std"
      )
      .reset_index()
)

# Save new CSV
bias_variance_df.to_csv(SUMMARIES_DIR / "estimator_bias_variance.csv", index=False)

print(bias_variance_df)

In [ ]:
import pandas as pd
import numpy as np

# ============================================================
# Input files
# ============================================================
files = {
    "hit_count": SUMMARIES_DIR / "subject_parametric_fit_summary_hit_count.csv",
    "incidence_frequency": SUMMARIES_DIR / "subject_parametric_fit_summary_incidence_frequency.csv",
    "zipf_coverage_frequency": SUMMARIES_DIR / "subject_parametric_fit_summary_zipf_coverage_frequency.csv",
}


def compute_delta_aic_summary(df: pd.DataFrame, source_name: str) -> pd.DataFrame:
    # Keep only rows with valid subject, model, aic
    df = df.copy()
    df = df.dropna(subset=["subject", "model", "aic"])

   
    df["aic"] = pd.to_numeric(df["aic"], errors="coerce")
    df = df.dropna(subset=["aic"])

  
    best_aic_per_subject = df.groupby("subject")["aic"].min().rename("best_aic")

    
    df = df.merge(best_aic_per_subject, on="subject", how="left")

    
    df["delta_aic"] = df["aic"] - df["best_aic"]

    
    summary = (
        df.groupby("model")
          .agg(
              subjects_won=("delta_aic", lambda x: int(np.isclose(x, 0).sum())),
              mean_delta_aic=("delta_aic", "mean"),
              median_delta_aic=("delta_aic", "median"),
              min_delta_aic=("delta_aic", "min"),
              max_delta_aic=("delta_aic", "max"),
          )
          .reset_index()
    )

    # Add source column
    summary.insert(0, "source", source_name)

    return summary



all_summaries = []

for source_name, file_path in files.items():
    df = pd.read_csv(file_path)
    summary_df = compute_delta_aic_summary(df, source_name)
    all_summaries.append(summary_df)


combined_summary = pd.concat(all_summaries, ignore_index=True)


combined_summary = combined_summary.sort_values(
    by=["source", "subjects_won", "mean_delta_aic"],
    ascending=[True, False, True]
).reset_index(drop=True)

output_file = "delta_aic_summary_all_models.csv"
combined_summary.to_csv(output_file, index=False)

print(f"Saved combined summary to: {output_file}")
print(combined_summary)

In [ ]:
# ============================================================
#  Fitted Zipf-Mandelbrot Distribution Parameters per Subject
# ============================================================

PAPER_TABLES_DIR = SUMMARIES_DIR / "paper_tables"
PAPER_TABLES_DIR.mkdir(parents=True, exist_ok=True)

zipf_params_src = pd.read_csv(SUMMARIES_DIR / "subject_parametric_fit_summary_zipf_coverage_frequency.csv")

table_zipf_params = (
    zipf_params_src[["subject", "C", "s", "q"]]
    .dropna(subset=["C", "s", "q"])
    .sort_values("subject")
    .reset_index(drop=True)
)

table_zipf_params.to_csv(PAPER_TABLES_DIR / "paper_table_zipf_mandelbrot_params.csv", index=False)
display(table_zipf_params)


In [ ]:
# ============================================================
#  Estimator Performance (Bias and Relative Error)
# ============================================================

PARAMETRIC_ESTIMATORS_PERF = ["Gamma-Poisson", "Zipf-Mandelbrot"]
NON_PARAMETRIC_ESTIMATORS_PERF = ["Chao2", "Chao2_bc", "Chiu", "Jackknife1", "Lanumteang-Bohning"]
ESTIMATOR_PERF_ORDER = PARAMETRIC_ESTIMATORS_PERF + NON_PARAMETRIC_ESTIMATORS_PERF

PAPER_TABLES_DIR = SUMMARIES_DIR / "paper_tables"
PAPER_TABLES_DIR.mkdir(parents=True, exist_ok=True)

bias_variance_src = pd.read_csv(SUMMARIES_DIR / "estimator_bias_variance.csv")
bias_variance_src = bias_variance_src[bias_variance_src["estimator"].isin(ESTIMATOR_PERF_ORDER)].copy()
bias_variance_src = bias_variance_src.rename(columns={"bias": "bias_mean", "variance": "bias_sd"})

by_subject_src = pd.read_csv(SUMMARIES_DIR / "estimator_summary_by_subject_known_totals.csv")
by_subject_src["estimator"] = by_subject_src["estimator_label"].str.replace(r"\s*\([^)]*\)\s*$", "", regex=True)
by_subject_src = by_subject_src[by_subject_src["estimator"].isin(ESTIMATOR_PERF_ORDER)]

rel_error_agg = (
    by_subject_src.groupby("estimator")["mean_rel_error"]
    .agg(rel_error_mean="mean", rel_error_sd="std")
    .reset_index()
)

table_estimator_performance = bias_variance_src.merge(rel_error_agg, on="estimator", how="left")
table_estimator_performance["group"] = np.where(
    table_estimator_performance["estimator"].isin(PARAMETRIC_ESTIMATORS_PERF),
    "Parametric",
    "Non-Parametric",
)
table_estimator_performance["estimator"] = pd.Categorical(
    table_estimator_performance["estimator"], categories=ESTIMATOR_PERF_ORDER, ordered=True
)
table_estimator_performance = table_estimator_performance.sort_values("estimator").reset_index(drop=True)
table_estimator_performance = table_estimator_performance[
    ["group", "estimator", "bias_mean", "bias_sd", "rel_error_mean", "rel_error_sd"]
]

table_estimator_performance.to_csv(PAPER_TABLES_DIR / "paper_table_estimator_performance.csv", index=False)
display(table_estimator_performance)


In [ ]:
# ============================================================
#Per-subject estimator comparison for synthetic parser benchmarks
# ============================================================

SUBJECT_ESTIMATOR_TABLE_ESTIMATORS = [
    "Gamma-Poisson (incidence-frequency estimator)",
    "Zipf-Mandelbrot (coverage-frequency estimator)",
    "Chao2 (incidence-based)",
    "Chao2_bc (incidence-based)",
    "Jackknife1 (incidence-based)",
    "Chiu (incidence-based)",
    "Lanumteang-Bohning (incidence-based)",
]

PAPER_TABLES_DIR = SUMMARIES_DIR / "paper_tables"
PAPER_TABLES_DIR.mkdir(parents=True, exist_ok=True)

comparison_known_src = pd.read_csv(SUMMARIES_DIR / "estimator_comparison_known_totals_only.csv")
subject_table_src = comparison_known_src[
    comparison_known_src["estimator_label"].isin(SUBJECT_ESTIMATOR_TABLE_ESTIMATORS)
].copy()

subject_mean_rel_error = subject_table_src.groupby("subject")["rel_error"].mean().sort_values()
lowest_subject = subject_mean_rel_error.index[0]
highest_subject = subject_mean_rel_error.index[-1]
print(f"Lowest relative-error subject: {lowest_subject} ({subject_mean_rel_error.iloc[0]:.4f})")
print(f"Highest relative-error subject: {highest_subject} ({subject_mean_rel_error.iloc[-1]:.4f})")

table_subject_estimator = subject_table_src[
    subject_table_src["subject"].isin([lowest_subject, highest_subject])
][["subject", "estimator_label", "known_total", "observed_total", "estimate_total", "rmse", "rel_error"]].copy()

table_subject_estimator["estimator_label"] = pd.Categorical(
    table_subject_estimator["estimator_label"], categories=SUBJECT_ESTIMATOR_TABLE_ESTIMATORS, ordered=True
)
table_subject_estimator["subject"] = pd.Categorical(
    table_subject_estimator["subject"], categories=[lowest_subject, highest_subject], ordered=True
)
table_subject_estimator = table_subject_estimator.sort_values(["subject", "estimator_label"]).reset_index(drop=True)
table_subject_estimator = table_subject_estimator.rename(columns={"rmse": "std_dev"})

table_subject_estimator.to_csv(PAPER_TABLES_DIR / "paper_table_subject_estimator_comparison.csv", index=False)
display(table_subject_estimator)


In [ ]:

PARAMETRIC_ESTIMATORS_PERF = ["Gamma-Poisson", "Zipf-Mandelbrot"]
NON_PARAMETRIC_ESTIMATORS_PERF = ["Chao2", "Chao2_bc", "Chiu", "Jackknife1", "Lanumteang-Bohning"]
ESTIMATOR_PERF_ORDER = PARAMETRIC_ESTIMATORS_PERF + NON_PARAMETRIC_ESTIMATORS_PERF

PAPER_TABLES_DIR = SUMMARIES_DIR / "paper_tables"
PAPER_TABLES_DIR.mkdir(parents=True, exist_ok=True)

real_world_overall_src = pd.read_csv(SUMMARIES_DIR / "estimator_relative_behavior_overall_unknown_totals.csv")

real_world_overall_src = real_world_overall_src.copy()
real_world_overall_src["estimator"] = real_world_overall_src["estimator_label"].str.replace(
    r"\s*\([^)]*\)\s*$", "", regex=True
)
real_world_overall_src = real_world_overall_src[
    real_world_overall_src["estimator"].isin(ESTIMATOR_PERF_ORDER)
].copy()

if real_world_overall_src.empty:
    print("No real-world rows found.")

table_real_world_estimator_performance = real_world_overall_src[
    ["estimator", "n", "mean_estimate", "median_estimate", "std_estimate",
     "min_estimate", "max_estimate", "mean_unseen_vs_observed"]
].copy()

table_real_world_estimator_performance["group"] = np.where(
    table_real_world_estimator_performance["estimator"].isin(PARAMETRIC_ESTIMATORS_PERF),
    "Parametric",
    "Non-Parametric",
)
table_real_world_estimator_performance["estimator"] = pd.Categorical(
    table_real_world_estimator_performance["estimator"], categories=ESTIMATOR_PERF_ORDER, ordered=True
)
table_real_world_estimator_performance = table_real_world_estimator_performance.sort_values("estimator").reset_index(drop=True)
table_real_world_estimator_performance = table_real_world_estimator_performance[
    ["group", "estimator", "n", "mean_estimate", "median_estimate", "std_estimate",
     "min_estimate", "max_estimate", "mean_unseen_vs_observed"]
]

table_real_world_estimator_performance.to_csv(
    PAPER_TABLES_DIR / "paper_table_real_world_estimator_performance.csv", index=False
)
display(table_real_world_estimator_performance)


In [ ]:

PAPER_TABLES_DIR = SUMMARIES_DIR / "paper_tables"
PAPER_TABLES_DIR.mkdir(parents=True, exist_ok=True)

SUBJECT_ESTIMATOR_TABLE_ESTIMATORS = [
    "Gamma-Poisson (incidence-frequency estimator)",
    "Zipf-Mandelbrot (coverage-frequency estimator)",
    "Chao2 (incidence-based)",
    "Chao2_bc (incidence-based)",
    "Jackknife1 (incidence-based)",
    "Chiu (incidence-based)",
    "Lanumteang-Bohning (incidence-based)",
]

comparison_unknown_src = pd.read_csv(SUMMARIES_DIR / "estimator_comparison_unknown_totals_only.csv")
real_world_table_src = comparison_unknown_src[
    comparison_unknown_src["estimator_label"].isin(SUBJECT_ESTIMATOR_TABLE_ESTIMATORS)
].copy()

if real_world_table_src.empty:
    print("No real-world  subject data found.")

all_real_world_subjects = sorted(real_world_table_src["subject"].unique())

table_real_world_subject_estimator = real_world_table_src[
    ["subject", "estimator_label", "observed_total", "estimate_total",
     "estimated_unseen_vs_observed", "n_campaigns"]
].copy()

table_real_world_subject_estimator["estimator_label"] = pd.Categorical(
    table_real_world_subject_estimator["estimator_label"], categories=SUBJECT_ESTIMATOR_TABLE_ESTIMATORS, ordered=True
)
table_real_world_subject_estimator["subject"] = pd.Categorical(
    table_real_world_subject_estimator["subject"], categories=all_real_world_subjects, ordered=True
)
table_real_world_subject_estimator = table_real_world_subject_estimator.sort_values(
    ["subject", "estimator_label"]
).reset_index(drop=True)

table_real_world_subject_estimator.to_csv(
    PAPER_TABLES_DIR / "paper_table_real_world_subject_estimator_comparison_all_subjects.csv", index=False
)
display(table_real_world_subject_estimator)


In [ ]:

import gc

ZIPF_ASC_DATA_DIR = SUMMARIES_DIR / "plot_data" / "zipf_rank_frequency_ascending"
ZIPF_ASC_DATA_DIR.mkdir(parents=True, exist_ok=True)

PARSER_LETTER = {f"parser{c}": c.upper() for c in "abcdefg"}

zipf_rank_frequency = {}
for subject, artifact in subject_artifacts.items():
    if subject not in PARSER_LETTER:
        continue

    incidence = artifact["incidence"]
    n_intervals = artifact["metadata"]["n_intervals"]
    axis = 0 if incidence.shape[0] == n_intervals else 1
    freq = incidence.sum(axis=axis)

    nonzero = freq[freq > 0]
    sorted_freq = np.sort(nonzero).astype(int)
    ranks = np.arange(1, sorted_freq.size + 1)

    out = pd.DataFrame({"rank": ranks, "frequency": sorted_freq})
    out.to_csv(ZIPF_ASC_DATA_DIR / f"{subject}_rank_frequency_ascending.csv", index=False)
    zipf_rank_frequency[subject] = out

    print(
        f"{subject} (parser{PARSER_LETTER[subject]}): "
        f"S_obs={sorted_freq.size}, J={int(sorted_freq.sum())}, max_freq={int(sorted_freq.max())}"
    )

    del freq, nonzero
    gc.collect()


In [ ]:

import gc

ZIPF_ASC_DATA_DIR_REAL = SUMMARIES_DIR / "plot_data" / "zipf_rank_frequency_ascending_real_world"
ZIPF_ASC_DATA_DIR_REAL.mkdir(parents=True, exist_ok=True)

def _real_world_display_name(subject: str) -> str:
    lowercase_as_is = {"libxml2", "zlib"}
    return subject if subject.lower() in lowercase_as_is else subject.capitalize()

real_world_subject_rows = subject_artifact_index_df[
    ~subject_artifact_index_df["subject"].isin(KNOWN_TOTALS.keys())
]

real_world_subject_artifacts = {
    row["subject"]: load_subject_artifact(row["artifact_dir"])
    for _, row in real_world_subject_rows.iterrows()
}
print(f"Loaded real-world subject artifacts: {len(real_world_subject_artifacts)}")

zipf_rank_frequency_real_world = {}
for subject, artifact in real_world_subject_artifacts.items():
    incidence = artifact["incidence"]
    n_intervals = artifact["metadata"]["n_intervals"]
    axis = 0 if incidence.shape[0] == n_intervals else 1
    freq = incidence.sum(axis=axis)

    nonzero = freq[freq > 0]
    sorted_freq = np.sort(nonzero).astype(int)
    ranks = np.arange(1, sorted_freq.size + 1)

    out = pd.DataFrame({"rank": ranks, "frequency": sorted_freq})
    out.to_csv(ZIPF_ASC_DATA_DIR_REAL / f"{subject}_rank_frequency_ascending.csv", index=False)
    zipf_rank_frequency_real_world[subject] = out

    display_name = _real_world_display_name(subject)
    print(
        f"{subject} ({display_name}): "
        f"S_obs={sorted_freq.size}, J={int(sorted_freq.sum())}, max_freq={int(sorted_freq.max())}"
    )

    del freq, nonzero
    gc.collect()


In [ ]:

import math

CSV_TABLES_DIR = Path("csv_tables")
CSV_TABLES_DIR.mkdir(parents=True, exist_ok=True)

TARGET_POINTS = 500 

def _downsample(ranks: np.ndarray, freqs: np.ndarray, target: int = TARGET_POINTS):
    n = len(ranks)
    stride = max(1, round(n / target))
    idx = np.arange(0, n, stride)
    if idx[-1] != n - 1:
        idx = np.append(idx, n - 1)
    return ranks[idx], freqs[idx]

def _nice_xmax(s_obs: int) -> int:
    xmax = math.ceil(s_obs / 1000) * 1000
    if xmax <= s_obs:
        xmax += 1000
    return xmax

def _xticks(xmax: int):
    half = round((xmax / 2) / 1000) * 1000 or 500
    def _lbl(v):
        return f"{v // 1000}k" if v % 1000 == 0 else str(v)
    return (half, xmax), (_lbl(half), _lbl(xmax))

panel_info = {}
for subject, df in zipf_rank_frequency.items():
    label = PARSER_LETTER[subject]
    ranks = df["rank"].to_numpy()
    freqs = df["frequency"].to_numpy()
    d_ranks, d_freqs = _downsample(ranks, freqs)

    downsampled_df = pd.DataFrame({"rank": d_ranks, "frequency": d_freqs})
    downsampled_df.to_csv(CSV_TABLES_DIR / f"zipf_rank_frequency_downsampled_parser{label}.csv", index=False)

    s_obs = int(ranks.max())
    xmax = _nice_xmax(s_obs)
    xtick, xticklabels = _xticks(xmax)
    panel_info[label] = {"subject": subject, "S_obs": s_obs, "xmax": xmax, "xtick": xtick, "xticklabels": xticklabels}

print("Panels generated:", sorted(panel_info.keys()))


In [ ]:

CSV_TABLES_DIR = Path("csv_tables")
CSV_TABLES_DIR.mkdir(parents=True, exist_ok=True)

panel_info_real_world = {}
for subject, df in zipf_rank_frequency_real_world.items():
    display_name = _real_world_display_name(subject)
    ranks = df["rank"].to_numpy()
    freqs = df["frequency"].to_numpy()
    d_ranks, d_freqs = _downsample(ranks, freqs)

    downsampled_df = pd.DataFrame({"rank": d_ranks, "frequency": d_freqs})
    downsampled_df.to_csv(CSV_TABLES_DIR / f"zipf_rank_frequency_downsampled_{subject}.csv", index=False)

    s_obs = int(ranks.max())
    xmax = _nice_xmax(s_obs)
    xtick, xticklabels = _xticks(xmax)
    panel_info_real_world[subject] = {
        "display_name": display_name, "S_obs": s_obs, "xmax": xmax, "xtick": xtick, "xticklabels": xticklabels
    }

print("Real-world panels generated:", sorted(panel_info_real_world.keys()))


In [ ]:


CSV_TABLES_DIR = Path("csv_tables")
CSV_TABLES_DIR.mkdir(parents=True, exist_ok=True)

AVAILABLE_LABELS = sorted(panel_info.keys())  # e.g. ['A', 'B', 'D', 'E', 'F']
N_COLS = 3
n_rows = math.ceil(len(AVAILABLE_LABELS) / N_COLS)

grid = [[None] * N_COLS for _ in range(n_rows)]
for i, label in enumerate(AVAILABLE_LABELS):
    grid[i // N_COLS][i % N_COLS] = label

# bottommost populated row per column -> that panel gets visible x tick labels / xlabel
bottom_row_of_col = {}
for c in range(N_COLS):
    for r in range(n_rows - 1, -1, -1):
        if grid[r][c] is not None:
            bottom_row_of_col[c] = r
            break

rows = []
for r in range(n_rows):
    for c in range(N_COLS):
        label = grid[r][c]
        if label is None:
            continue
        info = panel_info[label]
        t1, t2 = info["xtick"]
        l1, l2 = info["xticklabels"]
        rows.append({
            "label": label,
            "subject": info["subject"],
            "row": r,
            "col": c,
            "S_obs": info["S_obs"],
            "xmax": info["xmax"],
            "xtick_half": t1,
            "xtick_max": t2,
            "xticklabel_half": l1,
            "xticklabel_max": l2,
            "is_bottom_of_col": bottom_row_of_col[c] == r,
            "is_top_left": (r == 0 and c == 0),
        })

panel_layout_df = pd.DataFrame(rows).sort_values(["row", "col"]).reset_index(drop=True)
panel_layout_df.to_csv(CSV_TABLES_DIR / "zipf_panel_layout.csv", index=False)

missing_labels = sorted(set("ABCDEFG") - set(AVAILABLE_LABELS))
if missing_labels:
    print(f"NOTE: parser(s) {', '.join(missing_labels)} have no subject artifact data available "
          f"and are omitted ({len(AVAILABLE_LABELS)} of 7 panels populated).")
else:
    print("All 7 synthetic parsers are populated.")

display(panel_layout_df)


In [ ]:


CSV_TABLES_DIR = Path("csv_tables")
CSV_TABLES_DIR.mkdir(parents=True, exist_ok=True)

AVAILABLE_REAL_WORLD = sorted(panel_info_real_world.keys())
N_COLS_REAL = 3
n_rows_real = math.ceil(len(AVAILABLE_REAL_WORLD) / N_COLS_REAL) if AVAILABLE_REAL_WORLD else 0

grid_real = [[None] * N_COLS_REAL for _ in range(n_rows_real)]
for i, subject in enumerate(AVAILABLE_REAL_WORLD):
    grid_real[i // N_COLS_REAL][i % N_COLS_REAL] = subject

bottom_row_of_col_real = {}
for c in range(N_COLS_REAL):
    for r in range(n_rows_real - 1, -1, -1):
        if grid_real[r][c] is not None:
            bottom_row_of_col_real[c] = r
            break

rows = []
for r in range(n_rows_real):
    for c in range(N_COLS_REAL):
        subject = grid_real[r][c]
        if subject is None:
            continue
        info = panel_info_real_world[subject]
        t1, t2 = info["xtick"]
        l1, l2 = info["xticklabels"]
        rows.append({
            "subject": subject,
            "display_name": info["display_name"],
            "row": r,
            "col": c,
            "S_obs": info["S_obs"],
            "xmax": info["xmax"],
            "xtick_half": t1,
            "xtick_max": t2,
            "xticklabel_half": l1,
            "xticklabel_max": l2,
            "is_bottom_of_col": bottom_row_of_col_real[c] == r,
            "is_top_left": (r == 0 and c == 0),
        })

panel_layout_real_world_df = (
    pd.DataFrame(rows).sort_values(["row", "col"]).reset_index(drop=True)
    if rows
    else pd.DataFrame(columns=[
        "subject", "display_name", "row", "col", "S_obs", "xmax",
        "xtick_half", "xtick_max", "xticklabel_half", "xticklabel_max",
        "is_bottom_of_col", "is_top_left",
    ])
)
panel_layout_real_world_df.to_csv(CSV_TABLES_DIR / "zipf_panel_layout_real_world.csv", index=False)
display(panel_layout_real_world_df)


In [ ]:

import math
import matplotlib.pyplot as plt

ZIPF_PLOTS_DIR = Path("zipf_plots")
ZIPF_PLOTS_DIR.mkdir(parents=True, exist_ok=True)

def _plot_zipf_ascending(ax, ranks, freqs, xmax, title):
    ax.scatter(ranks, freqs, s=3, color="#3060cc", alpha=0.45, edgecolors="none")
    ax.set_xlim(0, xmax)
    ax.set_ylim(0, 700)
    ax.grid(True, which="major", linewidth=0.4, color="0.75")
    ax.set_title(title, fontsize=9)

def _combined_grid_plot(items, csv_name_fn, out_path, n_cols=3):
    """items: list of (key, info_dict) with info_dict having 'xmax' and a display title."""
    if not items:
        print(f"Nothing to plot for {out_path.name}; skipping.")
        return
    n_rows = math.ceil(len(items) / n_cols)
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(3 * n_cols, 2.4 * n_rows), squeeze=False)
    for i, (key, info, title) in enumerate(items):
        r, c = divmod(i, n_cols)
        d = pd.read_csv(csv_name_fn(key))
        _plot_zipf_ascending(axes[r][c], d["rank"], d["frequency"], info["xmax"], title)
    for i in range(len(items), n_rows * n_cols):
        r, c = divmod(i, n_cols)
        axes[r][c].axis("off")
    fig.supxlabel("Rank")
    fig.supylabel("Frequency")
    fig.tight_layout()
    fig.savefig(out_path, dpi=200)
    plt.close(fig)

# ---- per-parser (synthetic) ----
for label, info in panel_info.items():
    d = pd.read_csv(CSV_TABLES_DIR / f"zipf_rank_frequency_downsampled_parser{label}.csv")
    fig, ax = plt.subplots(figsize=(3.2, 2.6))
    _plot_zipf_ascending(ax, d["rank"], d["frequency"], info["xmax"], f"parser{label}")
    ax.set_xlabel("Rank")
    ax.set_ylabel("Frequency")
    fig.tight_layout()
    fig.savefig(ZIPF_PLOTS_DIR / f"parser{label}_zipf_ascending.png", dpi=200)
    plt.close(fig)

synth_items = [
    (label, panel_info[label], f"parser{label}")
    for label in sorted(panel_info.keys())
]
_combined_grid_plot(
    synth_items,
    lambda label: CSV_TABLES_DIR / f"zipf_rank_frequency_downsampled_parser{label}.csv",
    ZIPF_PLOTS_DIR / "zipf_synthetic_combined.png",
)
print(f"Saved {len(panel_info)} per-parser PNGs + combined PNG to {ZIPF_PLOTS_DIR}/")

# ---- per-real-world-program ----
for subject, info in panel_info_real_world.items():
    d = pd.read_csv(CSV_TABLES_DIR / f"zipf_rank_frequency_downsampled_{subject}.csv")
    fig, ax = plt.subplots(figsize=(3.2, 2.6))
    _plot_zipf_ascending(ax, d["rank"], d["frequency"], info["xmax"], info["display_name"])
    ax.set_xlabel("Rank")
    ax.set_ylabel("Frequency")
    fig.tight_layout()
    fig.savefig(ZIPF_PLOTS_DIR / f"{subject}_zipf_ascending.png", dpi=200)
    plt.close(fig)

real_world_items = [
    (subject, panel_info_real_world[subject], panel_info_real_world[subject]["display_name"])
    for subject in sorted(panel_info_real_world.keys())
]
_combined_grid_plot(
    real_world_items,
    lambda subject: CSV_TABLES_DIR / f"zipf_rank_frequency_downsampled_{subject}.csv",
    ZIPF_PLOTS_DIR / "zipf_real_world_combined.png",
)
if real_world_items:
    print(f"Saved {len(panel_info_real_world)} per-real-world-program PNGs + combined PNG to {ZIPF_PLOTS_DIR}/")


In [ ]:

import math

PAPER_TABLES_DIR = SUMMARIES_DIR / "paper_tables"
PAPER_TABLES_DIR.mkdir(parents=True, exist_ok=True)

MODEL_ORDER_SYNTH = [
    "Exponential (incidence-frequency)",
    "Gamma (incidence-frequency)",
    "Gamma-Poisson (incidence-frequency)",
    "Negative Binomial (incidence-frequency)",
    "Poisson (incidence-frequency)",
    "Zipf-Mandelbrot (coverage-frequency)",
]
MODEL_MACRO_SYNTH = {
    "Exponential (incidence-frequency)": r"\Exponential",
    "Gamma (incidence-frequency)": "Gamma",
    "Gamma-Poisson (incidence-frequency)": r"\GammaPoisson",
    "Negative Binomial (incidence-frequency)": r"\NBin",
    "Poisson (incidence-frequency)": "Poisson",
    "Zipf-Mandelbrot (coverage-frequency)": r"\Zipf",
}
SUBJECT_LETTER_SYNTH = {f"parser{c}": c.upper() for c in "abcdefg"}

fit_src = pd.read_csv(SUMMARIES_DIR / "subject_parametric_fit_summary.csv")
table_model_fit_synthetic = fit_src[fit_src["model"].isin(MODEL_ORDER_SYNTH)][
    ["subject", "model", "aic", "bic", "loglik"]
].copy()
table_model_fit_synthetic["model"] = pd.Categorical(
    table_model_fit_synthetic["model"], categories=MODEL_ORDER_SYNTH, ordered=True
)
table_model_fit_synthetic = table_model_fit_synthetic.sort_values(["subject", "model"]).reset_index(drop=True)

table_model_fit_synthetic.to_csv(PAPER_TABLES_DIR / "paper_table_model_fit_synthetic.csv", index=False)
display(table_model_fit_synthetic)


In [ ]:


CSV_TABLES_DIR = Path("csv_tables")
CSV_TABLES_DIR.mkdir(parents=True, exist_ok=True)

annotated_rows = []
for subject in sorted(table_model_fit_synthetic["subject"].unique()):
    sub = (
        table_model_fit_synthetic[table_model_fit_synthetic["subject"] == subject]
        .set_index("model")
        .loc[MODEL_ORDER_SYNTH]
    )
    best_aic_model = sub["aic"].idxmin()
    best_bic_model = sub["bic"].idxmin()
    best_loglik_model = sub["loglik"].idxmax()

    for model in MODEL_ORDER_SYNTH:
        r = sub.loc[model]
        annotated_rows.append({
            "subject": subject,
            "model": model,
            "aic": r["aic"],
            "bic": r["bic"],
            "loglik": r["loglik"],
            "is_best_aic": model == best_aic_model,
            "is_best_bic": model == best_bic_model,
            "is_best_loglik": model == best_loglik_model,
        })

paper_table_model_fit_synthetic_annotated = pd.DataFrame(annotated_rows)
paper_table_model_fit_synthetic_annotated.to_csv(
    CSV_TABLES_DIR / "paper_table_model_fit_synthetic_annotated.csv", index=False
)
display(paper_table_model_fit_synthetic_annotated)


In [ ]:

PAPER_TABLES_DIR = SUMMARIES_DIR / "paper_tables"
zipf_params_table = pd.read_csv(PAPER_TABLES_DIR / "paper_table_zipf_mandelbrot_params.csv")
zipf_params_table = zipf_params_table.sort_values("subject").reset_index(drop=True)

SUBJECT_LETTER_ZIPF = {f"parser{c}": c.upper() for c in "abcdefg"}
zipf_params_table["label"] = zipf_params_table["subject"].map(SUBJECT_LETTER_ZIPF)

CSV_TABLES_DIR = Path("csv_tables")
CSV_TABLES_DIR.mkdir(parents=True, exist_ok=True)
zipf_params_table.to_csv(CSV_TABLES_DIR / "paper_table_zipf_mandelbrot_params.csv", index=False)
display(zipf_params_table)


In [ ]:

PAPER_TABLES_DIR = SUMMARIES_DIR / "paper_tables"
PAPER_TABLES_DIR.mkdir(parents=True, exist_ok=True)

SUBJECT_ESTIMATOR_TABLE_ESTIMATORS = [
    "Gamma-Poisson (incidence-frequency estimator)",
    "Zipf-Mandelbrot (coverage-frequency estimator)",
    "Chao2 (incidence-based)",
    "Chao2_bc (incidence-based)",
    "Jackknife1 (incidence-based)",
    "Chiu (incidence-based)",
    "Lanumteang-Bohning (incidence-based)",
]
SUBJECT_LETTER_EST = {f"parser{c}": c.upper() for c in "abcdefg"}

comparison_known_src = pd.read_csv(SUMMARIES_DIR / "estimator_comparison_known_totals_only.csv")
subject_table_src = comparison_known_src[
    comparison_known_src["estimator_label"].isin(SUBJECT_ESTIMATOR_TABLE_ESTIMATORS)
].copy()

all_subjects = sorted(subject_table_src["subject"].unique(), key=lambda s: SUBJECT_LETTER_EST[s])

table_subject_estimator_all = subject_table_src[
    ["subject", "estimator_label", "known_total", "observed_total", "estimate_total", "rmse", "rel_error"]
].copy()
table_subject_estimator_all["estimator_label"] = pd.Categorical(
    table_subject_estimator_all["estimator_label"], categories=SUBJECT_ESTIMATOR_TABLE_ESTIMATORS, ordered=True
)
table_subject_estimator_all["subject"] = pd.Categorical(
    table_subject_estimator_all["subject"], categories=all_subjects, ordered=True
)
table_subject_estimator_all = table_subject_estimator_all.sort_values(["subject", "estimator_label"]).reset_index(drop=True)
table_subject_estimator_all = table_subject_estimator_all.rename(columns={"rmse": "std_dev"})

table_subject_estimator_all.to_csv(
    PAPER_TABLES_DIR / "paper_table_subject_estimator_comparison_all_subjects.csv", index=False
)
display(table_subject_estimator_all)


In [ ]:

CSV_TABLES_DIR = Path("csv_tables")
CSV_TABLES_DIR.mkdir(parents=True, exist_ok=True)

table_subject_estimator_all_out = table_subject_estimator_all.copy()
table_subject_estimator_all_out["label"] = table_subject_estimator_all_out["subject"].map(SUBJECT_LETTER_EST)
table_subject_estimator_all_out.to_csv(
    CSV_TABLES_DIR / "paper_table_subject_estimator_comparison_all_subjects.csv", index=False
)
display(table_subject_estimator_all_out)


In [ ]:


PAPER_TABLES_DIR = SUMMARIES_DIR / "paper_tables"
perf_df = pd.read_csv(PAPER_TABLES_DIR / "paper_table_estimator_performance.csv")

# Global best (closest-to-zero / lowest) value per metric, across ALL estimators (both groups).
best_bias_mean = perf_df["bias_mean"].abs().min()
best_bias_sd = perf_df["bias_sd"].min()
best_rel_mean = perf_df["rel_error_mean"].min()
best_rel_sd = perf_df["rel_error_sd"].min()

tol = 1e-9
perf_annotated = perf_df.copy()
perf_annotated["is_best_bias_mean"] = (perf_annotated["bias_mean"].abs() - best_bias_mean).abs() < tol
perf_annotated["is_best_bias_sd"] = (perf_annotated["bias_sd"] - best_bias_sd).abs() < tol
perf_annotated["is_best_rel_error_mean"] = (perf_annotated["rel_error_mean"] - best_rel_mean).abs() < tol
perf_annotated["is_best_rel_error_sd"] = (perf_annotated["rel_error_sd"] - best_rel_sd).abs() < tol

CSV_TABLES_DIR = Path("csv_tables")
CSV_TABLES_DIR.mkdir(parents=True, exist_ok=True)
perf_annotated.to_csv(CSV_TABLES_DIR / "paper_table_estimator_performance_annotated.csv", index=False)
display(perf_annotated)


In [ ]:
import json, glob, os

for f in sorted(glob.glob("parsers/grammar_*.json")):
    grammar = json.load(open(f))
    nt = len(grammar)
    rules = sum(len(alts) for alts in grammar.values())
    print(os.path.basename(f), "-> nonterminals:", nt, "| rules:", rules)